<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);border-radius:16px;padding:36px 40px;color:#ffffff;font-family:Calibri,'Segoe UI',sans-serif;">
  <div style="font-size:12px;font-weight:700;letter-spacing:3px;text-transform:uppercase;color:#F5C242;">01 &middot; HANDS-ON &middot; DIGITAL DEVELOPMENT DATA</div>
  <div style="font-size:42px;font-weight:700;line-height:1.08;margin-top:12px;">Ookla at Scale with Elasticsearch</div>
  <div style="font-size:17px;font-style:italic;color:#E6F6EE;margin-top:10px;max-width:820px;">
    Index Ookla performance tiles, understand the mapping and the <code style="color:#F5C242;background:rgba(255,255,255,.08);padding:1px 6px;border-radius:4px;">geo_shape</code> type, cross connectivity with WorldPop population, then publish a presentation-ready Kibana dashboard.
  </div>
  <div style="height:6px;width:140px;background:#F5C242;border-radius:3px;margin-top:26px;"></div>
</div>

<div style="display:flex;gap:14px;margin-top:18px;font-family:Calibri,'Segoe UI',sans-serif;flex-wrap:wrap;">
  <div style="flex:1;min-width:210px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Environments</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">Google Colab &middot; Kaggle &middot; Local (Linux / WSL / Docker)</div>
  </div>
  <div style="flex:1;min-width:210px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Estimated time</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">45 to 70 minutes (including ~8 min of install)</div>
  </div>
  <div style="flex:1;min-width:210px;background:#E8F5EF;border:1.5px solid #00A86A;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Deliverable</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">2 Elasticsearch indices + a 13-panel Kibana dashboard with a working time filter</div>
  </div>
</div>

## Contents

| # | Section | What you do |
|---|---|---|
| **00** | Parameters & style | Pick the country, the quarter, the stack version |
| **01** | Environment | Detect Colab / Kaggle / local, install dependencies |
| **02** | Elasticsearch + Kibana cluster | Download, configure, start, health check |
| **03** | Quadkey theory | Web Mercator quadtree, prefixes, lexicographic ordering |
| **04** | Extent & download | geoBoundaries ADM0/ADM1, DuckDB extraction over **N quarters**, spatial clip |
| **05** | WorldPop population | 1 km raster, per-tile density, zonal statistics |
| **06** | Mapping & indexing | `geo_shape`, `geo_point`, BKD-tree, `_bulk` |
| **07** | Aggregations | `stats`, `percentiles`, `weighted_avg`, **`date_histogram`**, `geotile_grid`, `geo_bounding_box`, `geo_shape`, `geo_distance` |
| **08** | Digital divide index | Composite metric, written to an analysis index |
| **09** | Kibana dashboard | 13 panels generated from code, time filter active, failure-tolerant import |
| **10** | Access, export, cleanup | Public URL, reusable NDJSON, shutting the stack down |

> **A note on styling** &mdash; this notebook follows the *AfDB-inspired Institutional* style guide (green `#00A86A`, deep green `#00704A`, gold `#F5C242`). It is **not** the official African Development Bank Group brand&nbsp;: confirm colours, fonts and logo with the communication department before any official use.

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">00 &middot; PARAMETERS</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">The only cell you need to edit</div>
</div>

The country is a **parameter**&nbsp;: change `COUNTRY_ISO3` and run the notebook end to end.
Examples&nbsp;: `TUN`, `MAR`, `DZA`, `EGY`, `SEN`, `CIV`, `GHA`, `NGA`, `KEN`, `RWA`, `ZAF`, `ETH`.

In [ ]:
# =============================================================================
#  NOTEBOOK PARAMETERS
# =============================================================================
COUNTRY_ISO3   = "TUN"            # ISO 3166-1 alpha-3 code of the country studied
YEAR           = 2024             # Most recent quarter: year
QUARTER        = 2                # Most recent quarter: quarter (1..4)
N_QUARTERS     = 4                # Consecutive quarters to load (this creates the time axis)
NETWORK_TYPES  = ["fixed", "mobile"]   # fixed and/or mobile
WORLDPOP_YEAR  = 2020             # WorldPop vintage (1 km UN-adjusted product)

# ---- Elastic stack -----------------------------------------------------------
INSTALL_STACK  = True             # False if you already have a reachable cluster
ES_VERSION     = "8.15.3"         # Version of BOTH Elasticsearch and Kibana
ES_HOST, ES_PORT = "127.0.0.1", 9200
KIBANA_PORT    = 5601
ES_HEAP        = "1g"             # JVM heap size (1g is plenty for this lab)

# ---- Safety rails ------------------------------------------------------------
MAX_TILES_PER_PERIOD = 500_000    # Cap per (network, quarter)
BULK_CHUNK           = 2_000      # Indexing batch size

# ---- Index names -------------------------------------------------------------
ISO = COUNTRY_ISO3.upper()
INDEX_TILES = f"ookla-tiles-{ISO.lower()}"   # 1 document = 1 Ookla tile (~610 m)
INDEX_ADMIN = f"ookla-admin-{ISO.lower()}"   # 1 document = 1 region x network x quarter

ES_URL     = f"http://{ES_HOST}:{ES_PORT}"
KIBANA_URL = f"http://127.0.0.1:{KIBANA_PORT}"

# =============================================================================
#  STYLE GUIDE  —  "AfDB-inspired Institutional"
# =============================================================================
AFDB = {
    "green":  "#00A86A",  # Primary accent
    "deep":   "#00704A",  # Kickers, secondary accent
    "forest": "#00553A",  # Gradient start, darkest bar
    "gold":   "#F5C242",  # Warm accent (heroes, section numbers)
    "ochre":  "#D49A00",  # Warnings, callouts on white
    "teal":   "#0E7C86",  # Third categorical colour
    "terra":  "#C4621D",  # Fourth categorical colour
    "brick":  "#B83B2E",  # Risks, negative values
    "ink":    "#231F20",  # Body text
    "slate":  "#5E6964",  # Captions, axis labels
    "mist":   "#F4F7F5",  # Card fill
    "mint":   "#E8F5EF",  # Highlighted card fill
    "sage":   "#D5DED9",  # Borders, dividers
    "grid":   "#E1E7E4",  # Very light gridlines
}
RAMP = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
SECTION_COLORS = [AFDB["green"], AFDB["deep"], AFDB["teal"],
                  AFDB["ochre"], AFDB["terra"], AFDB["brick"]]

# Speed classes used throughout (numeric prefix guarantees ordering in Kibana)
SPEED_CLASSES = [
    (0,    10,   "1 · < 10 Mbps"),
    (10,   25,   "2 · 10–25 Mbps"),
    (25,   100,  "3 · 25–100 Mbps"),
    (100,  1e9,  "4 · ≥ 100 Mbps"),
]

DASHBOARD_TITLE = f"[{ISO}] Ookla Connectivity x Population — {N_QUARTERS} quarters"
print(f"Country: {ISO}   |   {N_QUARTERS} quarters up to {YEAR} Q{QUARTER}"
      f"   |   Indices: {INDEX_TILES}, {INDEX_ADMIN}")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">01 &middot; ENVIRONMENT</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Platform detection and dependencies</div>
</div>

The notebook adapts on its own&nbsp;:

| Platform | Cluster | Kibana access |
|---|---|---|
| **Google Colab** | Tarball installed in the VM | Colab port proxy (public URL generated automatically) |
| **Kaggle** | Tarball installed in the VM (*Internet* must be enabled) | Optional `cloudflared` tunnel, or NDJSON export |
| **Local / WSL** | Tarball or existing cluster | `http://localhost:5601` |
| **Docker** | Set `ES_URL` / `KIBANA_URL` and `INSTALL_STACK = False` | Your own URL |

In [ ]:
import os, sys, json, time, math, shutil, subprocess, textwrap, platform, getpass
from pathlib import Path

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working") and not IN_COLAB
PLATFORM  = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else f"Local ({platform.system()})")

WORKDIR = Path("/content/work" if IN_COLAB else ("/kaggle/temp/work" if IN_KAGGLE else "./ookla_work")).resolve()
DATADIR = WORKDIR / "data"; STACKDIR = WORKDIR / "stack"; OUTDIR = WORKDIR / "out"
for d in (DATADIR, STACKDIR, OUTDIR):
    d.mkdir(parents=True, exist_ok=True)

IS_LINUX = platform.system() == "Linux"
try:
    IS_ROOT = (os.geteuid() == 0)
except AttributeError:      # Windows
    IS_ROOT = False

print(f"Platform       : {PLATFORM}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Working dir    : {WORKDIR}")
print(f"Linux / root   : {IS_LINUX} / {IS_ROOT}")
if INSTALL_STACK and not IS_LINUX:
    print("\n[!] Tarball installation is Linux-only.")
    print("    On macOS/Windows: run Elastic via Docker, then set INSTALL_STACK = False.")

In [ ]:
# -----------------------------------------------------------------------------
#  Install Python dependencies (only the missing ones)
# -----------------------------------------------------------------------------
REQUIRED = [
    ("elasticsearch>=8.10,<9", "elasticsearch"),
    ("duckdb>=0.10",           "duckdb"),
    ("pandas",                 "pandas"),
    ("numpy",                  "numpy"),
    ("pyarrow",                "pyarrow"),
    ("requests",               "requests"),
    ("matplotlib",             "matplotlib"),
    ("shapely>=2.0",           "shapely"),
    ("geopandas>=0.14",        "geopandas"),
    ("rasterio>=1.3",          "rasterio"),
    ("tqdm",                   "tqdm"),
]

import importlib
missing = []
for spec, mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(spec)

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--disable-pip-version-check", *missing], check=False)
else:
    print("All dependencies are already present.")

import numpy as np, pandas as pd, requests
import matplotlib as mpl, matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
from tqdm.auto import tqdm

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("Imports OK —", ", ".join(f"{m}={importlib.import_module(m).__version__}"
                                for m in ("pandas", "numpy")))

In [ ]:
# -----------------------------------------------------------------------------
#  Presentation helpers: matplotlib style + HTML cards (AfDB style guide)
# -----------------------------------------------------------------------------
def afdb_style():
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Calibri", "Carlito", "DejaVu Sans"],
        "font.size": 10.5,
        "text.color": AFDB["ink"], "axes.labelcolor": AFDB["slate"],
        "axes.edgecolor": AFDB["sage"], "axes.linewidth": 0.8,
        "xtick.color": AFDB["slate"], "ytick.color": AFDB["slate"],
        "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
        "axes.grid": True, "grid.color": AFDB["grid"], "grid.linewidth": 0.7,
        "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 14,
        "figure.dpi": 110, "legend.frameon": False,
    })
afdb_style()

def chart(ax, kicker, title, source=None):
    """Apply the AfDB chart template: small-caps kicker + italic source line."""
    ax.set_title("")
    ax.text(0, 1.14, kicker.upper(), transform=ax.transAxes, fontsize=9,
            fontweight="bold", color=AFDB["deep"])
    ax.text(0, 1.045, title, transform=ax.transAxes, fontsize=13.5,
            fontweight="bold", color=AFDB["ink"])
    if source:
        ax.text(0, -0.17, source, transform=ax.transAxes, fontsize=8.5,
                style="italic", color=AFDB["slate"])
    return ax

def card(title, body, tone="neutral"):
    """Display an HTML callout (neutral / success / warning / risk)."""
    fill, border, label = {
        "neutral": (AFDB["mist"], AFDB["sage"],  AFDB["deep"]),
        "ok":      (AFDB["mint"], AFDB["green"], AFDB["deep"]),
        "warn":    ("#FDF4E0",    AFDB["ochre"], AFDB["ochre"]),
        "risk":    ("#FBEDEB",    AFDB["brick"], AFDB["brick"]),
    }[tone]
    display(HTML(f"""
    <div style="background:{fill};border:1.4px solid {border};border-radius:12px;
                padding:14px 18px;margin:8px 0;font-family:Calibri,'Segoe UI',sans-serif;">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:{label};">{title}</div>
      <div style="font-size:14px;color:{AFDB['ink']};margin-top:6px;line-height:1.5;">{body}</div>
    </div>"""))

def kpi_row(items):
    """items = [(value, label, colour), ...]"""
    cells = "".join(f"""
      <div style="flex:1;min-width:150px;background:#fff;border:1px solid {AFDB['sage']};
                  border-left:5px solid {c};border-radius:12px;padding:16px 18px;">
        <div style="font-size:30px;font-weight:700;color:{c};line-height:1;">{v}</div>
        <div style="font-size:11.5px;color:{AFDB['slate']};margin-top:8px;">{lbl}</div>
      </div>""" for v, lbl, c in items)
    display(HTML(f"<div style=\"display:flex;gap:12px;flex-wrap:wrap;font-family:Calibri,'Segoe UI',sans-serif;margin:10px 0;\">{cells}</div>"))

def fmt(n, unit=""):
    if n is None or (isinstance(n, float) and math.isnan(n)): return "—"
    if abs(n) >= 1e9: return f"{n/1e9:,.1f} B{unit}"
    if abs(n) >= 1e6: return f"{n/1e6:,.1f} M{unit}"
    if abs(n) >= 1e3: return f"{n/1e3:,.1f} k{unit}"
    return f"{n:,.1f}{unit}"

card("Style guide applied",
     "Palette, typography and chart templates follow the <b>AfDB-inspired Institutional</b> "
     "guide &mdash; AfDB green <code>#00A86A</code>, deep green <code>#00704A</code>, "
     "gold <code>#F5C242</code>.", tone="ok")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">02 &middot; CLUSTER</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Provisioning Elasticsearch and Kibana</div>
</div>

We bring up a **single-node** cluster with security disabled&nbsp;: this is *workshop* mode, never production mode.

**Configuration choices, and why&nbsp;:**

| Setting | Value | Reason |
|---|---|---|
| `discovery.type` | `single-node` | No master election, immediate startup |
| `xpack.security.enabled` | `false` | No TLS or passwords to manage in a training room |
| `ES_HEAP` | `1g` | Colab/Kaggle offer ~13 GB of RAM; 1 GB of heap is enough |
| `ingest.geoip.downloader.enabled` | `false` | Avoids a pointless download at startup |
| `xpack.ml.enabled` | `false` | Frees memory |
| `network.host` | `127.0.0.1` | **Critical.** A non-loopback interface flips Elasticsearch into *production mode* and enables bootstrap checks (`vm.max_map_count` at 262144, and others) that cannot be satisfied inside a Colab container |
| `-Des.cgroups.hierarchy.override=/` | JVM option | **Critical.** Containers mount cgroup statistics at the root while leaving cgroup paths untouched; without this property `OsProbe` raises an `AccessControlException` and the node dies at startup. Elastic's official Docker image does exactly the same thing |

> **If the node refuses to start** &mdash; the `diagnose()` helper pulls the lines containing `ERROR`, `Caused by`, `access denied` or `bootstrap check` out of the log, instead of printing a truncated tail. Elasticsearch **9.x** replaces the Java `SecurityManager` with the entitlements system and can no longer raise an `AccessControlException`&nbsp;: if you still hit this class of error, moving `ES_VERSION` to 9.x is an alternative.

> **What about the "L" in ELK?** Logstash is not needed here&nbsp;: ingestion goes through the `_bulk` API from Python, which is easier to observe and debug in a workshop. Note also that Elasticsearch refuses to run as `root`&nbsp;: the notebook therefore creates a dedicated `esuser` when running as root (Colab/Kaggle).

In [ ]:
# -----------------------------------------------------------------------------
#  System helpers: download with progress bar + shell execution
# -----------------------------------------------------------------------------
RUN_USER = "esuser" if (IS_ROOT and IS_LINUX) else None

def sh(cmd, as_user=None, check=False, capture=False):
    """Run a shell command, optionally as another user."""
    if as_user:
        cmd = f"su {as_user} -s /bin/bash -c {json.dumps(cmd)}"
    return subprocess.run(cmd, shell=True, check=check,
                          stdout=subprocess.PIPE if capture else None,
                          stderr=subprocess.STDOUT if capture else None,
                          text=True)

def download(url, dest, desc=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  (cached) {dest.name}  —  {dest.stat().st_size/1e6:,.1f} MB")
        return dest
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                         desc=desc or dest.name, leave=False) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); bar.update(len(chunk))
    return dest

def service_up(url, path="/", timeout=2):
    try:
        return requests.get(url.rstrip("/") + path, timeout=timeout).status_code < 500
    except Exception:
        return False

def wait_for(check_fn, label, max_wait=300, every=5):
    t0 = time.time()
    with tqdm(total=max_wait, desc=f"Starting {label}", unit="s", leave=False) as bar:
        while time.time() - t0 < max_wait:
            if check_fn():
                bar.update(max_wait - bar.n)
                print(f"  {label} is up after {time.time()-t0:,.0f} s")
                return True
            time.sleep(every); bar.update(every)
    print(f"  [!] {label} did not start within {max_wait} s")
    return False

print("System helpers ready.")

In [ ]:
# -----------------------------------------------------------------------------
#  Download and configure Elasticsearch and Kibana
# -----------------------------------------------------------------------------
ES_HOME = STACKDIR / f"elasticsearch-{ES_VERSION}"
KB_HOME = STACKDIR / f"kibana-{ES_VERSION}"
ARTIFACTS = "https://artifacts.elastic.co/downloads"

# network.host stays on LOOPBACK: as soon as Elasticsearch listens on a non-loopback
# interface it switches to "production mode" and enables bootstrap checks
# (vm.max_map_count, file descriptors...) that cannot be satisfied inside a
# Colab/Kaggle container. Only Kibana needs to be reachable from outside.
ES_YML = textwrap.dedent(f"""
    cluster.name: ookla-workshop
    node.name: node-1
    network.host: 127.0.0.1
    http.port: {ES_PORT}
    discovery.type: single-node
    xpack.security.enabled: false
    xpack.security.enrollment.enabled: false
    xpack.ml.enabled: false
    ingest.geoip.downloader.enabled: false
    bootstrap.memory_lock: false
    action.destructive_requires_name: false
""").strip() + "\n"

# Colab/Kaggle are containers: the cgroup mount is rearranged there, which makes
# OsProbe.getCgroup() fail at startup (AccessControlException). Elastic provides the
# es.cgroups.hierarchy.override property for exactly this case — it is what the
# official Docker image's entrypoint does. The extra policy is a belt-and-braces
# grant on reading the pseudo filesystems.
ES_EXTRA_POLICY = (
    "grant {\n"
    '  permission java.io.FilePermission "/sys/fs/cgroup", "read";\n'
    '  permission java.io.FilePermission "/sys/fs/cgroup/-", "read";\n'
    '  permission java.io.FilePermission "/proc/-", "read";\n'
    "};\n"
)

KEY = "afdb_ookla_workshop_encryption_key_32+"
KB_YML = textwrap.dedent(f"""
    server.host: "0.0.0.0"
    server.port: {KIBANA_PORT}
    server.publicBaseUrl: "http://localhost:{KIBANA_PORT}"
    elasticsearch.hosts: ["http://127.0.0.1:{ES_PORT}"]
    telemetry.optIn: false
    telemetry.enabled: false
    xpack.encryptedSavedObjects.encryptionKey: "{KEY}"
    xpack.reporting.encryptionKey: "{KEY}"
    xpack.security.encryptionKey: "{KEY}"
    logging.root.level: warn
""").strip() + "\n"

def install_stack():
    if not IS_LINUX:
        raise RuntimeError("Tarball installation is Linux-only — use Docker instead.")
    if IS_ROOT:
        sh(f"id -u {RUN_USER} >/dev/null 2>&1 || useradd -m {RUN_USER}", capture=True)

    if not ES_HOME.exists():
        print("[1/4] Downloading Elasticsearch", ES_VERSION)
        tgz = download(f"{ARTIFACTS}/elasticsearch/elasticsearch-{ES_VERSION}-linux-x86_64.tar.gz",
                       STACKDIR / f"es-{ES_VERSION}.tar.gz", "elasticsearch")
        print("[2/4] Extracting..."); sh(f"tar -xzf {tgz} -C {STACKDIR}", check=True)
    if not KB_HOME.exists():
        print("[3/4] Downloading Kibana", ES_VERSION, "(~1 GB, be patient)")
        tgz = download(f"{ARTIFACTS}/kibana/kibana-{ES_VERSION}-linux-x86_64.tar.gz",
                       STACKDIR / f"kb-{ES_VERSION}.tar.gz", "kibana")
        print("[4/4] Extracting...")
        sh(f"tar -xzf {tgz} -C {STACKDIR}", check=True)
        extracted = STACKDIR / f"kibana-{ES_VERSION}-linux-x86_64"
        if extracted.exists() and not KB_HOME.exists():
            extracted.rename(KB_HOME)

    cfg = ES_HOME / "config"
    (cfg / "elasticsearch.yml").write_text(ES_YML)
    (cfg / "extra.policy").write_text(ES_EXTRA_POLICY)
    (cfg / "jvm.options.d").mkdir(exist_ok=True)
    (cfg / "jvm.options.d" / "workshop.options").write_text(
        "-Des.cgroups.hierarchy.override=/\n"
        f"-Djava.security.policy={cfg / 'extra.policy'}\n"
        f"-Xms{ES_HEAP}\n-Xmx{ES_HEAP}\n")
    (KB_HOME / "config" / "kibana.yml").write_text(KB_YML)
    if IS_ROOT:
        sh("sysctl -w vm.max_map_count=262144", capture=True)   # no-op if not permitted
        sh(f"chown -R {RUN_USER}:{RUN_USER} {STACKDIR}", capture=True)
    print("Stack installed under", STACKDIR)

if INSTALL_STACK and not service_up(ES_URL):
    install_stack()
elif service_up(ES_URL):
    print("A cluster already answers at", ES_URL, "— install skipped.")
else:
    print("INSTALL_STACK = False: the notebook will use", ES_URL)

In [ ]:
# -----------------------------------------------------------------------------
#  Starting the services
# -----------------------------------------------------------------------------
KEYWORDS = ("ERROR", "FATAL", "Caused by", "access denied", "bootstrap check",
            "uncaught exception", "fatal exception")

def diagnose(path, label):
    """Print the genuinely useful log lines, not a truncated tail."""
    if not Path(path).exists():
        print(f"  No {label} log found."); return
    txt = Path(path).read_text(errors="replace")
    hits = [l for l in txt.split("\n") if any(k in l for k in KEYWORDS)]
    print(f"\n--- {label} diagnostics ---")
    for l in (hits[:14] if hits else txt.split("\n")[-14:]):
        print("  " + l[:280])

def start_es():
    if service_up(ES_URL):
        print("Elasticsearch already running."); return True
    # Heap and JVM options live in config/jvm.options.d/workshop.options: we do not
    # also go through ES_JAVA_OPTS, to avoid setting the same thing twice.
    env = f"export ES_TMPDIR={STACKDIR}/tmp; mkdir -p {STACKDIR}/tmp; "
    sh(env + f"{ES_HOME}/bin/elasticsearch -d -p {STACKDIR}/es.pid", as_user=RUN_USER, capture=True)
    ok = wait_for(lambda: service_up(ES_URL), "Elasticsearch", max_wait=240)
    if not ok:
        logs = sorted((ES_HOME / "logs").glob("*.log"))
        diagnose(logs[-1] if logs else ES_HOME / "logs" / "absent.log", "Elasticsearch")
        print("\n  Things to check: network.host must stay 127.0.0.1,")
        print("  config/jvm.options.d/workshop.options must exist, and permissions on", STACKDIR)
    return ok

def start_kibana():
    if service_up(KIBANA_URL, "/api/status"):
        print("Kibana already running."); return True
    logf = STACKDIR / "kibana.log"
    sh(f"nohup {KB_HOME}/bin/kibana > {logf} 2>&1 &", as_user=RUN_USER, capture=True)
    def ready():
        try:
            s = requests.get(KIBANA_URL + "/api/status", timeout=3).json()
            return s.get("status", {}).get("overall", {}).get("level") == "available"
        except Exception:
            return False
    ok = wait_for(ready, "Kibana", max_wait=420, every=8)
    if not ok:
        diagnose(logf, "Kibana")
    return ok

if INSTALL_STACK or not service_up(ES_URL):
    if start_es():
        start_kibana()

if not service_up(ES_URL):
    card("Cluster unavailable",
         "Elasticsearch did not start. Read the diagnostics above before continuing&nbsp;: "
         "the usual causes are a non-loopback <code>network.host</code> (bootstrap checks) "
         "and cgroup detection inside a container.", tone="risk")
    raise SystemExit("Elasticsearch unavailable — fix this before continuing.")

info = requests.get(ES_URL, timeout=10).json()
kpi_row([
    (info["version"]["number"], "Elasticsearch version", AFDB["green"]),
    (info["cluster_name"],      "Cluster name",          AFDB["deep"]),
    (requests.get(f"{ES_URL}/_cluster/health", timeout=10).json()["status"].upper(),
                                "Cluster health",        AFDB["teal"]),
])

In [ ]:
# -----------------------------------------------------------------------------
#  Python client + cluster health check
# -----------------------------------------------------------------------------
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(ES_URL, request_timeout=180, retry_on_timeout=True, max_retries=3)
health = es.cluster.health()
print(json.dumps({k: health[k] for k in
      ("cluster_name", "status", "number_of_nodes", "active_shards", "unassigned_shards")},
      indent=2))

# The JVM version lives in _nodes/info, not in _nodes/stats (which only carries
# counters). nodes.info gives version AND heap size in a single call.
for n in es.nodes.info(metric="jvm")["nodes"].values():
    jvm = n["jvm"]
    print(f"Node {n['name']} — max heap {jvm['mem']['heap_max_in_bytes']/1e9:,.1f} GB, "
          f"JVM {jvm['version']} ({jvm.get('vm_name', '?')})")

# A "yellow" status is expected on a single node: Kibana's system indices ask for a
# replica that can never be allocated. Our own indices are created with replicas: 0.
if health["status"] == "yellow":
    print(f"\nYellow status: {health['unassigned_shards']} unassigned shards "
          "(system index replicas). Harmless for this lab.")
    # To force green: es.indices.put_settings(index="*", expand_wildcards="all",
    #                      settings={"index.number_of_replicas": 0})

### Open Kibana now

Kibana listens on port 5601 **inside the VM**, which is not exposed to the internet. Each platform has its own mechanism&nbsp;:

* **Colab** &mdash; `google.colab.kernel.proxyPort(5601)` generates a `googleusercontent.com` URL. It is tied to your Google account&nbsp;: open it in the **same browser profile**, otherwise you will get an authentication error.
* **Local** &mdash; `http://localhost:5601`, nothing to do.
* **Kaggle** &mdash; no exposed port&nbsp;: the cell suggests a free `cloudflared` tunnel.

Kibana's first startup takes 1 to 3 minutes. The cell waits for the status to reach `available` before handing you the URL.

In [ ]:
# -----------------------------------------------------------------------------
#  Access to the Kibana interface
# -----------------------------------------------------------------------------
def kibana_status():
    try:
        return requests.get(KIBANA_URL + "/api/status", timeout=5) \
                       .json()["status"]["overall"]["level"]
    except Exception:
        return None

def kibana_public_url(open_window=True):
    """Return a browser-openable URL, depending on the platform."""
    lvl = kibana_status()
    if lvl != "available":
        print(f"Kibana is not ready (status: {lvl}).")
        print("  -> run start_kibana() again, then re-run this cell.")
        print(f"  -> log: !tail -n 30 {STACKDIR}/kibana.log")
        return None
    if IN_COLAB:
        from google.colab.output import eval_js, serve_kernel_port_as_window
        url = eval_js(f"google.colab.kernel.proxyPort({KIBANA_PORT})").rstrip("/")
        if open_window:
            try: serve_kernel_port_as_window(KIBANA_PORT)
            except Exception: pass
        return url
    if IN_KAGGLE:
        print("Kaggle exposes no port. Free tunnel:")
        print("  !wget -q https://github.com/cloudflare/cloudflared/releases/latest/"
              "download/cloudflared-linux-amd64 -O /usr/local/bin/cf && chmod +x /usr/local/bin/cf")
        print(f"  !nohup cf tunnel --url http://localhost:{KIBANA_PORT} > /tmp/cf.log 2>&1 &")
        print("  !grep -o 'https://.*trycloudflare.com' /tmp/cf.log | head -1")
        return None
    return KIBANA_URL

KIBANA_PUBLIC = kibana_public_url()
if KIBANA_PUBLIC:
    display(HTML(f"""
    <div style="background:{AFDB['mint']};border:1.5px solid {AFDB['green']};border-radius:12px;
                padding:16px 20px;font-family:Calibri,'Segoe UI',sans-serif;">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;color:{AFDB['deep']};">KIBANA IS UP</div>
      <div style="font-size:14px;margin-top:8px;line-height:1.9;">
        <a href="{KIBANA_PUBLIC}" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Home</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/discover" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Discover</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/maps" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Maps</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/dashboards" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Dashboards</a>
      </div>
      <div style="font-size:12px;color:{AFDB['slate']};margin-top:8px;">
        Data views only exist after section 09. For now the cluster is empty.
      </div>
    </div>"""))
    print(KIBANA_PUBLIC)

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">03 &middot; OOKLA DATA</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">The quadkey, or how to download 0.3&nbsp;% of a 2&nbsp;GB file</div>
</div>

Every quarter Ookla publishes, as open data, performance aggregated by **zoom-16 Web Mercator tile** (&asymp;&nbsp;610&nbsp;m at the equator). One **global** Parquet file per quarter and per network type, i.e. several million rows.

**Source schema** &mdash; `quadkey`, `tile` (WKT polygon), `avg_d_kbps`, `avg_u_kbps`, `avg_lat_ms`, `tests`, `devices`.

### Why the quadkey changes everything

A *quadkey* is the base-4 encoding of a tile's path through the Web Mercator quadtree. Two properties follow&nbsp;:

1. **Prefix = spatial containment.** Zoom-4 tile `1202` contains every zoom-16 tile whose quadkey starts with `1202`. Filtering an area therefore becomes filtering **string prefixes**.
2. **Lexicographic order &asymp; spatial order** (Z-curve). Since the Parquet file is sorted by quadkey, the min/max statistics of each row group let DuckDB discard almost every block remotely&nbsp;: this is **predicate pushdown** over HTTP range requests.

The result&nbsp;: a few dozen megabytes downloaded instead of two gigabytes, with no intermediate server.

In [ ]:
# -----------------------------------------------------------------------------
#  Quadkey mathematics (Web Mercator / Bing Maps quadtree)
# -----------------------------------------------------------------------------
Z_OOKLA = 16   # zoom level of Ookla tiles

def lonlat_to_tile_xy(lon, lat, z):
    """(lon, lat) in degrees -> tile indices (x, y) at zoom z."""
    lat = max(min(lat, 85.05112878), -85.05112878)
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    s = math.sin(math.radians(lat))
    y = int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
    return min(max(x, 0), n - 1), min(max(y, 0), n - 1)

def tile_xy_to_quadkey(x, y, z):
    """Tile indices -> quadkey (string of z base-4 digits)."""
    out = []
    for i in range(z, 0, -1):
        digit, mask = 0, 1 << (i - 1)
        if x & mask: digit += 1
        if y & mask: digit += 2
        out.append(str(digit))
    return "".join(out)

def quadkey_prefixes_for_bbox(bbox, max_prefixes=96):
    """Smallest set of prefixes covering the bbox (minx, miny, maxx, maxy)."""
    minx, miny, maxx, maxy = bbox
    for z in range(10, 1, -1):
        x0, y0 = lonlat_to_tile_xy(minx, maxy, z)
        x1, y1 = lonlat_to_tile_xy(maxx, miny, z)
        n = (x1 - x0 + 1) * (y1 - y0 + 1)
        if n <= max_prefixes:
            return [tile_xy_to_quadkey(x, y, z)
                    for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)], z, n
    raise RuntimeError("Bounding box too large")

def _quadkey_succ(s):
    """Immediate successor of a quadkey in lexicographic (base-4) order."""
    d = list(s)
    for i in range(len(d) - 1, -1, -1):
        if d[i] != "3":
            d[i] = str(int(d[i]) + 1)
            return "".join(d[:i + 1]) + "0" * (len(d) - i - 1)
        d[i] = "0"
    return None

def quadkey_ranges(prefixes, z_target=Z_OOKLA):
    """Prefixes -> merged [lo, hi] intervals usable in a SQL BETWEEN.

    Prefixes that are contiguous along the Z-curve are also contiguous
    lexicographically: merging them sharply cuts the number of OR clauses
    (Tunisia, for instance: 91 -> 23).
    """
    rng = sorted((p + "0" * (z_target - len(p)), p + "3" * (z_target - len(p)))
                 for p in prefixes)
    merged = []
    for lo, hi in rng:
        if merged and (lo <= merged[-1][1] or lo == _quadkey_succ(merged[-1][1])):
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])
    return merged

def quadkeys_to_bounds(qk_series):
    """Vectorised: a Series of quadkeys -> (min_lon, min_lat, max_lon, max_lat) in degrees."""
    arr = np.asarray(qk_series, dtype=str)
    z = len(arr[0])
    digits = (np.frombuffer("".join(arr.tolist()).encode("ascii"), dtype=np.uint8)
                .reshape(-1, z).astype(np.int64) - 48)
    weights = (1 << np.arange(z - 1, -1, -1)).astype(np.int64)
    x = ((digits & 1) * weights).sum(axis=1)
    y = (((digits >> 1) & 1) * weights).sum(axis=1)
    n = float(1 << z)
    lon0 = x / n * 360.0 - 180.0
    lon1 = (x + 1) / n * 360.0 - 180.0
    lat0 = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * y / n))))
    lat1 = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * (y + 1) / n))))
    return lon0, np.minimum(lat0, lat1), lon1, np.maximum(lat0, lat1)

# Teaching check
demo_qk = tile_xy_to_quadkey(*lonlat_to_tile_xy(10.18, 36.80, Z_OOKLA), Z_OOKLA)  # Tunis
b = quadkeys_to_bounds(pd.Series([demo_qk]))
card("Sanity check",
     f"Tunis (10.18, 36.80) &rarr; quadkey <code>{demo_qk}</code> at zoom 16.<br>"
     f"Reconstructed extent: lon [{b[0][0]:.5f}, {b[2][0]:.5f}], lat [{b[1][0]:.5f}, {b[3][0]:.5f}]<br>"
     f"Zoom-6 prefix: <code>{demo_qk[:6]}</code> &mdash; it contains {4**10:,} zoom-16 tiles.")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">04 &middot; EXTENT & DOWNLOAD</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">geoBoundaries borders, then the quarterly series</div>
</div>

**geoBoundaries** (William & Mary, CC BY) provides ADM0 and ADM1. We use it for the **extent** (building the quadkey prefixes) and for the **exact clip** with regional attribution.

The difference from a one-off extraction&nbsp;: we download **`N_QUARTERS` consecutive quarters**. This is what gives Kibana's time picker something to do &mdash; with a single quarter every document carries the same date and the time filter filters nothing. The cost is proportional&nbsp;: four quarters means four DuckDB queries and four times as many documents.

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

GB_API = "https://www.geoboundaries.org/api/current/gbOpen/{iso}/{lvl}/"

def load_boundaries(iso3, level):
    """Fetch an administrative level from geoBoundaries (GeoJSON, EPSG:4326)."""
    meta = requests.get(GB_API.format(iso=iso3.upper(), lvl=level), timeout=90).json()
    if isinstance(meta, list): meta = meta[0]
    url = meta.get("gjDownloadURL") or meta.get("staticDownloadLink")
    gdf = gpd.read_file(url)
    if gdf.crs is None: gdf.set_crs(4326, inplace=True)
    return gdf.to_crs(4326), meta

adm0, meta0 = load_boundaries(ISO, "ADM0")
COUNTRY_NAME = meta0.get("boundaryName", ISO)

try:
    adm1, _ = load_boundaries(ISO, "ADM1")
    adm1 = adm1.rename(columns={"shapeName": "admin1"})[["admin1", "geometry"]]
    HAS_ADM1 = True
except Exception as exc:
    print("ADM1 unavailable:", exc)
    adm1 = adm0.assign(admin1=COUNTRY_NAME)[["admin1", "geometry"]]
    HAS_ADM1 = False

# Geometry cleanup (self-intersections are common in open datasets)
adm0["geometry"] = adm0.geometry.buffer(0)
adm1["geometry"] = adm1.geometry.buffer(0)
COUNTRY_GEOM = unary_union(adm0.geometry.values)
BBOX = tuple(adm0.total_bounds)

prefixes, z_pref, n_pref = quadkey_prefixes_for_bbox(BBOX)
RANGES = quadkey_ranges(prefixes)

kpi_row([
    (COUNTRY_NAME,            "Country",                   AFDB["green"]),
    (f"{len(adm1)}",          "ADM1 regions" if HAS_ADM1 else "ADM0 only", AFDB["deep"]),
    (f"{n_pref} / z{z_pref}", "Quadkey prefixes",          AFDB["teal"]),
    (f"{len(RANGES)}",        "SQL ranges after merging",  AFDB["ochre"]),
])
print(f"Bounding box: lon [{BBOX[0]:.3f}, {BBOX[2]:.3f}]  lat [{BBOX[1]:.3f}, {BBOX[3]:.3f}]")

In [ ]:
# -----------------------------------------------------------------------------
#  Remote extraction from the Ookla Parquet, quarter by quarter
# -----------------------------------------------------------------------------
import duckdb

QUARTER_MONTH = {1: "01", 2: "04", 3: "07", 4: "10"}

def quarters_back(year, quarter, n):
    """[(year, quarter)] for the n quarters ending at (year, quarter), ascending."""
    out, y, q = [], year, quarter
    for _ in range(n):
        out.append((y, q))
        q -= 1
        if q == 0: y, q = y - 1, 4
    return list(reversed(out))

PERIODS = quarters_back(YEAR, QUARTER, N_QUARTERS)
print("Target quarters:", ", ".join(f"{y} Q{q}" for y, q in PERIODS))

def ookla_url(network, year, quarter):
    return (f"https://ookla-open-data.s3.amazonaws.com/parquet/performance/"
            f"type={network}/year={year}/quarter={quarter}/"
            f"{year}-{QUARTER_MONTH[quarter]}-01_performance_{network}_tiles.parquet")

def fetch_ookla(network, year, quarter, ranges, limit=None):
    con = duckdb.connect()
    for stmt in ("INSTALL httpfs;", "LOAD httpfs;"):
        try: con.execute(stmt)
        except Exception: pass
    where = " OR ".join(f"(quadkey BETWEEN '{lo}' AND '{hi}')" for lo, hi in ranges)
    sql = f"""SELECT quadkey, avg_d_kbps, avg_u_kbps, avg_lat_ms, tests, devices
              FROM read_parquet('{ookla_url(network, year, quarter)}')
              WHERE {where} {f'LIMIT {limit}' if limit else ''}"""
    df = con.execute(sql).df()
    con.close()
    return df

raw = {}
for (y, q) in PERIODS:
    for net in NETWORK_TYPES:
        t0 = time.time()
        try:
            df = fetch_ookla(net, y, q, RANGES, limit=MAX_TILES_PER_PERIOD)
            raw[(net, y, q)] = df
            print(f"  {y} Q{q} · {net:<6} : {len(df):>8,} tiles  ({time.time()-t0:,.1f} s)")
        except Exception as exc:
            raw[(net, y, q)] = pd.DataFrame()
            print(f"  {y} Q{q} · {net:<6} : FAILED — {str(exc)[:110]}")

got = sorted({(y, q) for (n, y, q), v in raw.items() if len(v)})
assert got, "No Ookla data retrieved — check YEAR/QUARTER, N_QUARTERS and internet access."

lost = [f"{y} Q{q}" for (y, q) in PERIODS if (y, q) not in got]
kpi_row([
    (f"{len(got)}/{len(PERIODS)}",             "Quarters retrieved", AFDB["green"]),
    (f"{sum(len(v) for v in raw.values()):,}", "Raw tiles",          AFDB["deep"]),
    (f"{len({n for (n, _, _), v in raw.items() if len(v)})}", "Network types", AFDB["teal"]),
])
if lost:
    card("Missing quarters",
         f"No data for&nbsp;: <b>{', '.join(lost)}</b>. Ookla publishes with a lag of a few "
         "weeks&nbsp;; a quarter that is too recent simply does not exist yet. Move "
         "<code>QUARTER</code> back one step and re-run from the parameters cell.", tone="warn")

In [ ]:
# -----------------------------------------------------------------------------
#  Tile geometry, country clip, ADM1 attribution
# -----------------------------------------------------------------------------
def enrich(df, network, year, quarter):
    if df.empty: return df
    df = df.copy()
    lon0, lat0, lon1, lat1 = quadkeys_to_bounds(df["quadkey"])
    df["min_lon"], df["min_lat"], df["max_lon"], df["max_lat"] = lon0, lat0, lon1, lat1
    df["lon"], df["lat"] = (lon0 + lon1) / 2.0, (lat0 + lat1) / 2.0
    df["tile_km2"] = ((lon1 - lon0) * 111.320 * np.cos(np.radians(df["lat"]))) * \
                     ((lat1 - lat0) * 110.574)
    df["download_mbps"] = df["avg_d_kbps"] / 1000.0
    df["upload_mbps"]   = df["avg_u_kbps"] / 1000.0
    df["latency_ms"]    = df["avg_lat_ms"].astype(float)
    df["network"]       = network
    df["period"]        = f"{year}-{QUARTER_MONTH[quarter]}-01"
    df["year_quarter"]  = f"{year} Q{quarter}"   # categorical key: alpha order = chrono order
    return df

tiles = pd.concat([enrich(df, n, y, q) for (n, y, q), df in raw.items() if len(df)],
                  ignore_index=True)
print(f"Tiles before clipping: {len(tiles):,}")

pts = gpd.GeoDataFrame(tiles[["quadkey"]].copy(),
                       geometry=gpd.points_from_xy(tiles["lon"], tiles["lat"]), crs=4326)
joined = gpd.sjoin(pts, adm1[["admin1", "geometry"]], how="left", predicate="within")
joined = joined[~joined.index.duplicated(keep="first")].reindex(pts.index)
tiles["admin1"] = joined["admin1"].values

before = len(tiles)
tiles = tiles[tiles["admin1"].notna()].reset_index(drop=True)
print(f"Tiles kept within {COUNTRY_NAME}: {len(tiles):,}  "
      f"({before - len(tiles):,} dropped: sea, neighbouring countries)")

def speed_class(v):
    for lo, hi, label in SPEED_CLASSES:
        if lo <= v < hi: return label
    return SPEED_CLASSES[-1][2]

tiles["speed_class"] = tiles["download_mbps"].apply(speed_class)
QUARTER_LABELS = sorted(tiles["year_quarter"].unique())
LATEST_Q = QUARTER_LABELS[-1]
display(tiles.head(4)[["quadkey", "year_quarter", "network", "admin1", "download_mbps",
                       "latency_ms", "tests", "speed_class"]])

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">05 &middot; POPULATION</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Crossing connectivity with WorldPop</div>
</div>

An Ookla tile tells you how fast people connect, **not how many people are concerned**. WorldPop (*Global 1 km, UN-adjusted*) provides a grid where each pixel carries a headcount.

Two distinct calculations&nbsp;:

* **Population per tile** &mdash; the pixel value under the tile centre, converted to **density (people/km&sup2;)** by dividing by the pixel's real area at that latitude, then multiplied by the tile's area. This is an **estimate**.
* **Population per region** &mdash; a zonal statistic (sum of pixels inside the polygon). This is the denominator of the coverage rate.

Population is held constant over the period&nbsp;: WorldPop publishes this product annually, and change over a few quarters is negligible next to the model's own uncertainty.

In [ ]:
import rasterio
from rasterio.mask import mask as rio_mask

def worldpop_url(iso3, year):
    return (f"https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/"
            f"{year}/{iso3.upper()}/{iso3.lower()}_ppp_{year}_1km_Aggregated_UNadj.tif")

POP_TIF = DATADIR / f"worldpop_{ISO.lower()}_{WORLDPOP_YEAR}.tif"
HAS_POP = True
try:
    download(worldpop_url(ISO, WORLDPOP_YEAR), POP_TIF, "worldpop")
    with rasterio.open(POP_TIF) as src:
        print(f"Raster: {src.width} x {src.height} px, resolution {src.res[0]:.5f} deg, "
              f"CRS {src.crs}, nodata {src.nodata}")
except Exception as exc:
    HAS_POP = False
    print("[!] WorldPop unavailable:", exc)

In [ ]:
# -----------------------------------------------------------------------------
#  Estimated population per tile (vectorised raster read)
# -----------------------------------------------------------------------------
def attach_population(df, tif):
    with rasterio.open(tif) as src:
        arr = src.read(1).astype("float64")
        if src.nodata is not None: arr[arr == src.nodata] = np.nan
        arr[arr < 0] = np.nan
        rows, cols = rasterio.transform.rowcol(src.transform,
                                               df["lon"].values, df["lat"].values)
        rows, cols = np.asarray(rows), np.asarray(cols)
        ok = (rows >= 0) & (rows < arr.shape[0]) & (cols >= 0) & (cols < arr.shape[1])
        vals = np.full(len(df), np.nan)
        vals[ok] = arr[rows[ok], cols[ok]]
        res_x, res_y = src.res
    px_km2 = (res_x * 111.320 * np.cos(np.radians(df["lat"].values))) * (res_y * 110.574)
    density = vals / px_km2
    return np.nan_to_num(density * df["tile_km2"].values, nan=0.0), np.nan_to_num(density, nan=0.0)

if HAS_POP:
    tiles["population"], tiles["pop_density"] = attach_population(tiles, POP_TIF)
else:
    tiles["population"], tiles["pop_density"] = 0.0, 0.0

# Numeric prefixes guarantee category ordering in Kibana
tiles["settlement"] = np.where(tiles["pop_density"] >= 1500, "3 · Dense urban",
                       np.where(tiles["pop_density"] >= 300, "2 · Urban", "1 · Rural"))

latest = tiles[tiles["year_quarter"] == LATEST_Q]
kpi_row([
    (f"{len(tiles):,}",                          "Tiles, all quarters",         AFDB["green"]),
    (fmt(tiles["tests"].sum()),                  "Cumulative speed tests",      AFDB["deep"]),
    (fmt(latest["population"].sum()),            "Population under measured tiles", AFDB["teal"]),
    (f"{latest['download_mbps'].median():,.1f}", f"Median download, {LATEST_Q}", AFDB["ochre"]),
])

In [ ]:
# -----------------------------------------------------------------------------
#  Total population per region (zonal statistics)
# -----------------------------------------------------------------------------
def zonal_population(gdf, tif):
    out = {}
    with rasterio.open(tif) as src:
        for _, row in tqdm(list(gdf.iterrows()), desc="Zonal statistics", leave=False):
            try:
                data, _ = rio_mask(src, [row.geometry.__geo_interface__], crop=True, filled=True)
                a = data[0].astype("float64")
                if src.nodata is not None: a[a == src.nodata] = np.nan
                a[a < 0] = np.nan
                out[row["admin1"]] = float(np.nansum(a))
            except Exception:
                out[row["admin1"]] = float("nan")
    return out

ADMIN_POP = zonal_population(adm1, POP_TIF) if HAS_POP else {a: float("nan") for a in adm1["admin1"]}
pop_total = np.nansum(list(ADMIN_POP.values()))
print(f"Total population of {COUNTRY_NAME} ({WORLDPOP_YEAR}): {pop_total:,.0f} people")
display(pd.Series(ADMIN_POP, name="population").sort_values(ascending=False).head(8).to_frame())

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">06 &middot; MAPPING</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">The mapping, and why <code>geo_shape</code></div>
</div>

The *mapping* is the index schema&nbsp;: it fixes the type of every field **before** indexing. A download speed that ends up as `text` makes any numeric aggregation impossible without a full reindex.

### `geo_point` vs `geo_shape`

| | `geo_point` | `geo_shape` |
|---|---|---|
| Represents | one lat/lon pair | point, line, polygon, **envelope**, multi-geometries |
| Internal structure | 2D BKD-tree | BKD-tree over the **cells** the geometry is decomposed into |
| Queries | `geo_distance`, `geo_bounding_box`, `geo_grid` | `geo_shape` (`intersects`, `within`, `contains`, `disjoint`) |
| Aggregations | `geo_centroid`, `geohash_grid`, `geotile_grid`, `geo_bounds` | `geotile_grid`, `geo_bounds` |
| Cost | low | higher (indexing and storage) |

**Our choice&nbsp;: both.** `tile_geom` as a `geo_shape` of type **`envelope`** (two corners are enough for an axis-aligned rectangle&nbsp;: the most compact form for a Mercator tile) and `centroid` as a `geo_point` (essential for `geo_distance` and for the grid aggregations Kibana Maps relies on).

The **`period`** field is a `date`&nbsp;: it is what will drive Kibana's time picker. The `year_quarter` field is its `keyword` counterpart, handier for a categorical axis.

In [ ]:
# -----------------------------------------------------------------------------
#  Mapping of the tiles index
# -----------------------------------------------------------------------------
TILES_SETTINGS = {"number_of_shards": 1, "number_of_replicas": 0, "refresh_interval": "30s"}

TILES_MAPPINGS = {
    "dynamic": "strict",           # any undeclared field is rejected: a teaching guard rail
    "properties": {
        "quadkey":       {"type": "keyword"},
        "network":       {"type": "keyword"},
        "period":        {"type": "date", "format": "yyyy-MM-dd"},
        "year_quarter":  {"type": "keyword"},
        "country_iso3":  {"type": "keyword"},
        "country":       {"type": "keyword"},
        "admin1":        {"type": "keyword"},
        "settlement":    {"type": "keyword"},
        "speed_class":   {"type": "keyword"},

        "tile_geom":     {"type": "geo_shape"},      # <- the subject of this lab
        "centroid":      {"type": "geo_point"},

        "download_mbps": {"type": "float"},
        "upload_mbps":   {"type": "float"},
        "latency_ms":    {"type": "float"},
        "tests":         {"type": "integer"},
        "devices":       {"type": "integer"},
        "population":    {"type": "float"},
        "pop_density":   {"type": "float"},
        "tile_km2":      {"type": "float"},
    },
}

if es.indices.exists(index=INDEX_TILES):
    es.indices.delete(index=INDEX_TILES)
es.indices.create(index=INDEX_TILES, settings=TILES_SETTINGS, mappings=TILES_MAPPINGS)
print(f"Index '{INDEX_TILES}' created.")
display(HTML(f"<pre style='background:{AFDB['mist']};border:1px solid {AFDB['sage']};"
             f"border-radius:10px;padding:14px;font-size:12px;'>"
             + json.dumps(es.indices.get_mapping(index=INDEX_TILES)
                            .body[INDEX_TILES]["mappings"]["properties"]["tile_geom"], indent=2)
             + "</pre>"))

In [ ]:
# -----------------------------------------------------------------------------
#  Bulk indexing (_bulk)
# -----------------------------------------------------------------------------
def tile_actions(df):
    for r in df.to_dict("records"):
        yield {
            "_index": INDEX_TILES,
            "_id": f"{r['network']}_{r['period']}_{r['quadkey']}",
            "_source": {
                "quadkey": r["quadkey"], "network": r["network"],
                "period": r["period"], "year_quarter": r["year_quarter"],
                "country_iso3": ISO, "country": COUNTRY_NAME,
                "admin1": r["admin1"], "settlement": r["settlement"],
                "speed_class": r["speed_class"],
                # Envelope = [[top-left corner], [bottom-right corner]] as [lon, lat]
                "tile_geom": {"type": "envelope",
                              "coordinates": [[round(r["min_lon"], 6), round(r["max_lat"], 6)],
                                              [round(r["max_lon"], 6), round(r["min_lat"], 6)]]},
                "centroid": {"lat": round(r["lat"], 6), "lon": round(r["lon"], 6)},
                "download_mbps": round(float(r["download_mbps"]), 3),
                "upload_mbps":   round(float(r["upload_mbps"]), 3),
                "latency_ms":    round(float(r["latency_ms"]), 2),
                "tests": int(r["tests"]), "devices": int(r["devices"]),
                "population":  round(float(r["population"]), 2),
                "pop_density": round(float(r["pop_density"]), 2),
                "tile_km2":    round(float(r["tile_km2"]), 5),
            },
        }

t0, ok_count, errors = time.time(), 0, []
with tqdm(total=len(tiles), desc="Bulk indexing", unit="doc") as bar:
    for ok, item in helpers.streaming_bulk(es, tile_actions(tiles), chunk_size=BULK_CHUNK,
                                           max_retries=3, raise_on_error=False,
                                           request_timeout=180):
        ok_count += int(ok)
        if not ok: errors.append(item)
        bar.update(1)

es.indices.refresh(index=INDEX_TILES)
elapsed = max(time.time() - t0, 1e-9)
size = es.indices.stats(index=INDEX_TILES)["indices"][INDEX_TILES]["total"]["store"]["size_in_bytes"]

kpi_row([
    (f"{es.count(index=INDEX_TILES)['count']:,}", "Documents indexed",  AFDB["green"]),
    (f"{ok_count/elapsed:,.0f}",                  "Documents / second", AFDB["deep"]),
    (f"{size/1e6:,.1f} MB",                       "Index size",         AFDB["teal"]),
    (f"{len(errors):,}", "Errors", AFDB["brick"] if errors else AFDB["slate"]),
])
if errors:
    print("Example error:", json.dumps(errors[0], indent=2)[:600])

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">07 &middot; AGGREGATIONS</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Statistical, temporal and geospatial aggregations</div>
</div>

Every query below uses `size: 0`&nbsp;: we retrieve **no documents at all**, only the results the cluster computes. That is the normal way to work at scale&nbsp;: the data does not move, the computation goes to it.

The path&nbsp;: `stats` &rarr; `percentiles` &rarr; `weighted_avg` &rarr; `date_histogram` &rarr; nested `terms` &rarr; `filter` &rarr; `geotile_grid` &rarr; `geo_bounding_box` &rarr; `geo_shape` &rarr; `geo_distance`.

In [ ]:
# -----------------------------------------------------------------------------
#  7.1  Descriptive statistics and percentiles, by network type
# -----------------------------------------------------------------------------
PRIMARY_NET = "fixed" if "fixed" in set(tiles["network"]) else sorted(set(tiles["network"]))[0]

resp = es.search(index=INDEX_TILES, size=0, aggs={
    "by_network": {
        "terms": {"field": "network"},
        "aggs": {
            "speed":       {"stats": {"field": "download_mbps"}},
            "percentiles": {"percentiles": {"field": "download_mbps",
                                            "percents": [10, 25, 50, 75, 90, 95]}},
            "latency":     {"percentiles": {"field": "latency_ms", "percents": [50, 90]}},
            "tests":       {"sum": {"field": "tests"}},
            # Weighted by the number of tests: a tile with 3 tests should not carry
            # the same weight as a tile with 3,000.
            "weighted_speed": {"weighted_avg": {"value": {"field": "download_mbps"},
                                                "weight": {"field": "tests"}}},
        }}})

rows = []
for b in resp["aggregations"]["by_network"]["buckets"]:
    p = b["percentiles"]["values"]
    rows.append({"network": b["key"], "tiles": b["doc_count"], "tests": int(b["tests"]["value"]),
                 "simple_mean": b["speed"]["avg"], "weighted_mean": b["weighted_speed"]["value"],
                 "p10": p["10.0"], "median": p["50.0"], "p90": p["90.0"], "p95": p["95.0"],
                 "latency_p50": b["latency"]["values"]["50.0"]})
display(pd.DataFrame(rows).set_index("network").style.format("{:,.1f}").set_caption(
    f"Download speed (Mbps) and latency (ms) — {COUNTRY_NAME}, {len(QUARTER_LABELS)} quarters"))

card("How to read this",
     "The gap between the <b>simple mean</b> and the <b>test-weighted mean</b> measures a "
     "bias&nbsp;: when the weighted figure is clearly higher, the most heavily tested areas "
     "(the most urban ones) are also the best served.", tone="warn")

In [ ]:
# -----------------------------------------------------------------------------
#  7.2  date_histogram: Elasticsearch's native temporal aggregation
# -----------------------------------------------------------------------------
resp = es.search(index=INDEX_TILES, size=0, aggs={
    "quarters": {
        "date_histogram": {"field": "period", "calendar_interval": "quarter",
                           "min_doc_count": 1, "format": "yyyy-MM-dd"},
        "aggs": {
            "by_network": {
                "terms": {"field": "network"},
                "aggs": {"median": {"percentiles": {"field": "download_mbps", "percents": [50]}},
                         "tests":  {"sum": {"field": "tests"}},
                         "pop_10": {"filter": {"range": {"download_mbps": {"gte": 10}}},
                                    "aggs": {"pop": {"sum": {"field": "population"}}}},
                         "pop":    {"sum": {"field": "population"}}}}}}})

evo = []
for b in resp["aggregations"]["quarters"]["buckets"]:
    for nb in b["by_network"]["buckets"]:
        pop = nb["pop"]["value"] or np.nan
        evo.append({"period": b["key_as_string"], "network": nb["key"],
                    "tiles": nb["doc_count"],
                    "median_dl": nb["median"]["values"]["50.0"],
                    "tests": nb["tests"]["value"],
                    "pct_pop_10mbps": 100 * nb["pop_10"]["pop"]["value"] / pop})
evo = pd.DataFrame(evo)
display(evo.pivot(index="period", columns="network",
                  values="median_dl").style.format("{:,.1f}").set_caption(
                  "Median download speed (Mbps) by quarter"))

card("Why this matters",
     "<code>calendar_interval: quarter</code> lets Elasticsearch own the quarter "
     "boundaries&nbsp;: no client-side date arithmetic, no timezone mistakes. That same "
     "<code>period</code> field is what makes Kibana's time picker act on the whole "
     "dashboard.", tone="ok")

In [ ]:
# -----------------------------------------------------------------------------
#  Chart 1: quarterly evolution
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.4))
palette = {"fixed": AFDB["green"], "mobile": AFDB["teal"]}

for net, grp in evo.groupby("network"):
    g = grp.sort_values("period")
    axes[0].plot(g["period"], g["median_dl"], marker="o", linewidth=2.6, markersize=7,
                 color=palette.get(net, AFDB["deep"]), label=net)
    axes[1].plot(g["period"], g["pct_pop_10mbps"], marker="o", linewidth=2.6, markersize=7,
                 color=palette.get(net, AFDB["deep"]), label=net)

axes[0].set_ylabel("Mbps"); axes[0].legend()
chart(axes[0], "07 · TIME SERIES", "Median download speed by quarter",
      "Source: Ookla Open Data, date_histogram aggregation computed in Elasticsearch.")
axes[1].set_ylabel("% of measured population"); axes[1].legend()
chart(axes[1], "07 · COVERAGE", "Measured population above 10 Mbps",
      "Sources: Ookla Open Data, WorldPop. Author's calculation.")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------------------------------------------------------
#  7.3  Nested aggregation: regional ranking (latest quarter)
# -----------------------------------------------------------------------------
resp = es.search(index=INDEX_TILES, size=0,
    query={"bool": {"filter": [{"term": {"network": PRIMARY_NET}},
                               {"term": {"year_quarter": LATEST_Q}}]}},
    aggs={"regions": {
        "terms": {"field": "admin1", "size": 60, "order": {"median_dl.50": "desc"}},
        "aggs": {
            "median_dl":  {"percentiles": {"field": "download_mbps", "percents": [50]}},
            "median_up":  {"percentiles": {"field": "upload_mbps",  "percents": [50]}},
            "median_lat": {"percentiles": {"field": "latency_ms",   "percents": [50]}},
            "population": {"sum": {"field": "population"}},
            "tests":      {"sum": {"field": "tests"}},
            "pop_10":     {"filter": {"range": {"download_mbps": {"gte": 10}}},
                           "aggs": {"pop": {"sum": {"field": "population"}}}},
            "pop_100":    {"filter": {"range": {"download_mbps": {"gte": 100}}},
                           "aggs": {"pop": {"sum": {"field": "population"}}}},
        }}})

reg = pd.DataFrame([{
    "admin1": b["key"], "tiles": b["doc_count"], "tests": int(b["tests"]["value"]),
    "median_dl": b["median_dl"]["values"]["50.0"],
    "median_up": b["median_up"]["values"]["50.0"],
    "median_lat": b["median_lat"]["values"]["50.0"],
    "pop_measured": b["population"]["value"],
    "pop_10": b["pop_10"]["pop"]["value"], "pop_100": b["pop_100"]["pop"]["value"],
} for b in resp["aggregations"]["regions"]["buckets"]])
reg["pct_10mbps"]  = 100 * reg["pop_10"]  / reg["pop_measured"].replace(0, np.nan)
reg["pct_100mbps"] = 100 * reg["pop_100"] / reg["pop_measured"].replace(0, np.nan)
display(reg.head(12)[["admin1", "tiles", "median_dl", "median_up", "median_lat",
                      "pct_10mbps", "pct_100mbps"]])

In [ ]:
# -----------------------------------------------------------------------------
#  Chart 2: regional ranking and the effect of density
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(14, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1])
ax0, ax1 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

top = reg.dropna(subset=["median_dl"]).head(12).sort_values("median_dl")
colors = [RAMP[min(len(RAMP)-1, int(i / max(len(top)-1, 1) * (len(RAMP)-1)))]
          for i in range(len(top))]
bars = ax0.barh(top["admin1"], top["median_dl"], color=colors, edgecolor="none", height=0.68)
for b, v in zip(bars, top["median_dl"]):
    ax0.text(v + max(top["median_dl"]) * 0.012, b.get_y() + b.get_height()/2,
             f"{v:,.1f}", va="center", fontsize=9.5, fontweight="bold", color=AFDB["ink"])
ax0.set_xlabel("Median download speed (Mbps)")
ax0.xaxis.grid(True); ax0.yaxis.grid(False)
ax0.set_xlim(0, max(top["median_dl"]) * 1.16)
chart(ax0, f"07 · {PRIMARY_NET.upper()} · {LATEST_Q}", "Median download speed by region",
      "Source: Ookla Open Data, aggregated by Elasticsearch.")

sub = tiles[(tiles["network"] == PRIMARY_NET) & (tiles["year_quarter"] == LATEST_Q) &
            (tiles["pop_density"] > 0)]
sub = sub.sample(min(8000, len(sub)), random_state=7)
ax1.scatter(sub["pop_density"], sub["download_mbps"].clip(upper=400), s=6,
            alpha=0.25, color=AFDB["deep"], edgecolors="none")
ax1.set_xscale("log")
ax1.set_xlabel("Population density (people/km², log scale)"); ax1.set_ylabel("Speed (Mbps)")
chart(ax1, "07 · CORRELATION", "Settlement density and download speed",
      "Each dot is a ~610 m tile. Sources: Ookla Open Data, WorldPop.")
plt.tight_layout(); plt.show()

### 7.4 &mdash; **Geospatial** aggregations and queries

This is where the mapping pays off. Four complementary mechanisms&nbsp;:

* **`geotile_grid`** &mdash; aggregates on the Web Mercator grid itself. Keys are `z/x/y`&nbsp;: quadkey logic, server side. This is what powers Kibana Maps heat layers.
* **`geo_bounding_box`** &mdash; a rectangular filter on `geo_point`, very fast (bound comparisons inside the BKD-tree).
* **`geo_shape` + `relation`** &mdash; a filter against an arbitrary geometry&nbsp;: `intersects` (default), `within`, `contains`, `disjoint`.
* **`geo_distance`** &mdash; a circular filter around a point, essential for radius analysis.

In [ ]:
# -----------------------------------------------------------------------------
#  7.4.a  geotile_grid
# -----------------------------------------------------------------------------
GRID_Z = 8
resp = es.search(index=INDEX_TILES, size=0,
    query={"bool": {"filter": [{"term": {"network": PRIMARY_NET}},
                               {"term": {"year_quarter": LATEST_Q}}]}},
    aggs={"grid": {"geotile_grid": {"field": "centroid", "precision": GRID_Z, "size": 5000},
                   "aggs": {"speed": {"avg": {"field": "download_mbps"}},
                            "pop":   {"sum": {"field": "population"}}}}})

g = []
for b in resp["aggregations"]["grid"]["buckets"]:
    z, x, y = map(int, b["key"].split("/"))
    n = 2 ** z
    g.append({"lon": (x + 0.5) / n * 360.0 - 180.0,
              "lat": math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 0.5) / n)))),
              "tiles": b["doc_count"], "speed": b["speed"]["value"], "pop": b["pop"]["value"]})
grid = pd.DataFrame(g)
print(f"{len(grid)} cells at z{GRID_Z} (≈ {40075/2**GRID_Z:,.0f} km at the equator)")

fig, ax = plt.subplots(figsize=(7.4, 7.0))
adm1.boundary.plot(ax=ax, color=AFDB["sage"], linewidth=0.7)
sc = ax.scatter(grid["lon"], grid["lat"],
                s=np.clip(grid["tiles"] / grid["tiles"].max() * 420, 14, 420),
                c=grid["speed"], cmap=mpl.colors.LinearSegmentedColormap.from_list("afdb", RAMP),
                alpha=0.88, edgecolors="white", linewidths=0.4)
cb = plt.colorbar(sc, ax=ax, shrink=0.72, pad=0.02)
cb.set_label("Mean download speed (Mbps)", color=AFDB["slate"], fontsize=9.5)
cb.outline.set_edgecolor(AFDB["sage"])
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(alpha=0.35)
ax.set_aspect(1 / math.cos(math.radians(float(np.mean(grid["lat"])))))
chart(ax, f"07 · GEOTILE_GRID z{GRID_Z}", f"Server-side geospatial aggregation — {LATEST_Q}",
      "Circle size = number of tiles. Source: Ookla Open Data.")
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------------------------------------------------------
#  7.4.b  geo_bounding_box, geo_shape and geo_distance
# -----------------------------------------------------------------------------
last = tiles[(tiles["year_quarter"] == LATEST_Q) & (tiles["network"] == PRIMARY_NET)]
hub = last.loc[last["pop_density"].idxmax()] if HAS_POP else last.loc[last["tests"].idxmax()]
HUB_LAT, HUB_LON = float(hub["lat"]), float(hub["lon"])
print(f"Main urban centre detected: {hub['admin1']}  ({HUB_LAT:.4f}, {HUB_LON:.4f})")

D = 0.25
q_bbox = {"geo_bounding_box": {"centroid": {
    "top_left":     {"lat": HUB_LAT + D, "lon": HUB_LON - D},
    "bottom_right": {"lat": HUB_LAT - D, "lon": HUB_LON + D}}}}

top_region = reg.iloc[0]["admin1"] if len(reg) else adm1.iloc[0]["admin1"]
geom_region = adm1.loc[adm1["admin1"] == top_region, "geometry"].iloc[0].simplify(0.01).buffer(0)
q_shape = {"geo_shape": {"tile_geom": {
    "shape": json.loads(gpd.GeoSeries([geom_region]).to_json())["features"][0]["geometry"],
    "relation": "intersects"}}}

q_dist = {"geo_distance": {"distance": "25km", "centroid": {"lat": HUB_LAT, "lon": HUB_LON}}}

AGGS = {"speed": {"percentiles": {"field": "download_mbps", "percents": [50]}},
        "pop":   {"sum": {"field": "population"}},
        "tests": {"sum": {"field": "tests"}}}

results = []
for label, q in [("Whole country", {"match_all": {}}),
                 ("geo_bounding_box (±25 km of hub)", q_bbox),
                 (f"geo_shape intersects ({str(top_region)[:20]})", q_shape),
                 ("geo_distance 25 km from hub", q_dist)]:
    t0 = time.time()
    r = es.search(index=INDEX_TILES, size=0, aggs=AGGS,
                  query={"bool": {"filter": [q, {"term": {"network": PRIMARY_NET}},
                                             {"term": {"year_quarter": LATEST_Q}}]}})
    results.append({"query": label, "tiles": r["hits"]["total"]["value"],
                    "median_speed": r["aggregations"]["speed"]["values"]["50.0"],
                    "population": r["aggregations"]["pop"]["value"],
                    "tests": r["aggregations"]["tests"]["value"],
                    "ms": (time.time() - t0) * 1000})
display(pd.DataFrame(results).set_index("query").style.format(
    {"tiles": "{:,.0f}", "median_speed": "{:,.1f}", "population": "{:,.0f}",
     "tests": "{:,.0f}", "ms": "{:,.0f} ms"}))

card("What to take away",
     "<code>geo_bounding_box</code> and <code>geo_distance</code> work on the "
     "<code>geo_point</code>&nbsp;: fast, but they test the tile's <b>centre</b>. "
     "<code>geo_shape intersects</code> works on the real geometry&nbsp;: a tile straddling "
     "a regional border is counted. The right choice depends on the question you are "
     "asking, not on performance.", tone="ok")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">08 &middot; ANALYSIS</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Digital divide index and results index</div>
</div>

We now produce a **second index**, `ookla-admin-…`, at *region &times; network &times; quarter* granularity. Few documents, but rich ones, carrying the ADM1 geometry as a `geo_shape` **polygon** &mdash; which enables the choropleth layer in Kibana and tracking over time.

**Digital divide index (0 = no lag, 100 = maximum lag)**&nbsp;:

$$\text{DDI} = 100 \times \Big[\;0.50\,(1 - p_{\ge 10}) \;+\; 0.30\,\big(1 - \min(\tfrac{\tilde{d}}{50}, 1)\big) \;+\; 0.20\,(1 - c)\;\Big]$$

where $p_{\ge 10}$ is the share of measured population above 10&nbsp;Mbps, $\tilde{d}$ the median download speed and $c$ the coverage rate.

> **Methodological warning.** Ookla tiles only cover areas where tests were actually run&nbsp;: the absence of a tile means "no measurement", not "no network". The coverage rate should be read as an **observability** indicator. The weights are a workshop choice, to be calibrated with sector teams before any decision-making use.

In [ ]:
# -----------------------------------------------------------------------------
#  8.1  Region x network x quarter documents
# -----------------------------------------------------------------------------
from shapely.geometry import mapping as shp_mapping
from shapely.geometry.polygon import orient

GEOM_CACHE = {}
for _, row in adm1.iterrows():
    gm = row.geometry.simplify(0.005).buffer(0)
    GEOM_CACHE[row["admin1"]] = (
        shp_mapping(orient(gm, sign=1.0) if gm.geom_type == "Polygon" else gm),
        {"lat": round(float(gm.centroid.y), 6), "lon": round(float(gm.centroid.x), 6)})

def build_admin_docs():
    docs = []
    for (net, yq), grp_all in tiles.groupby(["network", "year_quarter"]):
        period = grp_all["period"].iloc[0]
        for adm, grp in grp_all.groupby("admin1"):
            pop_tot  = ADMIN_POP.get(adm, float("nan"))
            pop_meas = float(grp["population"].sum())
            p10  = float(grp.loc[grp["download_mbps"] >= 10,  "population"].sum())
            p25  = float(grp.loc[grp["download_mbps"] >= 25,  "population"].sum())
            p100 = float(grp.loc[grp["download_mbps"] >= 100, "population"].sum())
            med  = float(grp["download_mbps"].median())
            cov   = (pop_meas / pop_tot) if (pop_tot == pop_tot and pop_tot > 0) else np.nan
            pct10 = (p10 / pop_meas) if pop_meas > 0 else np.nan
            ddi = 100 * (0.50 * (1 - (pct10 if pct10 == pct10 else 0))
                       + 0.30 * (1 - min(med / 50.0, 1.0))
                       + 0.20 * (1 - (min(cov, 1.0) if cov == cov else 0)))
            geom, centroid = GEOM_CACHE[adm]
            docs.append({
                "admin1": adm, "network": net, "country": COUNTRY_NAME, "country_iso3": ISO,
                "period": period, "year_quarter": yq,
                "population": None if pop_tot != pop_tot else round(pop_tot, 0),
                "pop_measured": round(pop_meas, 0),
                "coverage_rate": None if cov != cov else round(float(cov) * 100, 2),
                "pop_above_10mbps": round(p10, 0),
                "pct_pop_10mbps":  None if pct10 != pct10 else round(pct10 * 100, 2),
                "pct_pop_25mbps":  round(100 * p25 / pop_meas, 2) if pop_meas > 0 else None,
                "pct_pop_100mbps": round(100 * p100 / pop_meas, 2) if pop_meas > 0 else None,
                "median_download_mbps": round(med, 2),
                "median_upload_mbps":   round(float(grp["upload_mbps"].median()), 2),
                "median_latency_ms":    round(float(grp["latency_ms"].median()), 2),
                "weighted_download_mbps": round(float(
                    np.average(grp["download_mbps"], weights=grp["tests"].clip(lower=1))), 2),
                "tiles": int(len(grp)), "tests": int(grp["tests"].sum()),
                "devices": int(grp["devices"].sum()),
                "digital_divide_index": round(float(ddi), 1),
                "centroid": centroid, "geometry": geom,
            })
    return docs

ADMIN_DOCS = build_admin_docs()
admin_df = pd.DataFrame([{k: v for k, v in d.items() if k not in ("geometry", "centroid")}
                         for d in ADMIN_DOCS])
print(f"{len(ADMIN_DOCS)} documents: {admin_df['admin1'].nunique()} regions x "
      f"{admin_df['network'].nunique()} networks x {admin_df['year_quarter'].nunique()} quarters")
display(admin_df[(admin_df["network"] == PRIMARY_NET) & (admin_df["year_quarter"] == LATEST_Q)]
        .sort_values("digital_divide_index", ascending=False)
        .head(10)[["admin1", "population", "coverage_rate", "median_download_mbps",
                   "pct_pop_10mbps", "digital_divide_index"]])

In [ ]:
# -----------------------------------------------------------------------------
#  8.2  Indexing the analysis results
# -----------------------------------------------------------------------------
ADMIN_MAPPINGS = {
    "dynamic": "strict",
    "properties": {
        "admin1": {"type": "keyword"}, "network": {"type": "keyword"},
        "country": {"type": "keyword"}, "country_iso3": {"type": "keyword"},
        "period": {"type": "date", "format": "yyyy-MM-dd"},
        "year_quarter": {"type": "keyword"},
        "geometry": {"type": "geo_shape"}, "centroid": {"type": "geo_point"},
        "population": {"type": "float"}, "pop_measured": {"type": "float"},
        "coverage_rate": {"type": "float"}, "pop_above_10mbps": {"type": "float"},
        "pct_pop_10mbps": {"type": "float"}, "pct_pop_25mbps": {"type": "float"},
        "pct_pop_100mbps": {"type": "float"},
        "median_download_mbps": {"type": "float"}, "median_upload_mbps": {"type": "float"},
        "median_latency_ms": {"type": "float"}, "weighted_download_mbps": {"type": "float"},
        "tiles": {"type": "integer"}, "tests": {"type": "long"}, "devices": {"type": "long"},
        "digital_divide_index": {"type": "float"},
    },
}

if es.indices.exists(index=INDEX_ADMIN):
    es.indices.delete(index=INDEX_ADMIN)
es.indices.create(index=INDEX_ADMIN,
                  settings={"number_of_shards": 1, "number_of_replicas": 0},
                  mappings=ADMIN_MAPPINGS)

ok, errs = helpers.bulk(
    es, ({"_index": INDEX_ADMIN,
          "_id": f"{d['network']}_{d['period']}_{d['admin1']}", "_source": d}
         for d in ADMIN_DOCS),
    raise_on_error=False, request_timeout=180)
es.indices.refresh(index=INDEX_ADMIN)

kpi_row([
    (f"{es.count(index=INDEX_TILES)['count']:,}", "Documents · tiles",    AFDB["green"]),
    (f"{es.count(index=INDEX_ADMIN)['count']:,}", "Documents · analysis", AFDB["deep"]),
    (f"{len(QUARTER_LABELS)}",                    "Quarters covered",     AFDB["teal"]),
    (f"{len(errs)}", "Errors", AFDB["brick"] if errs else AFDB["slate"]),
])
if errs: print(json.dumps(errs[0], indent=2)[:600])

In [ ]:
# -----------------------------------------------------------------------------
#  Chart 3: choropleth maps (latest quarter)
# -----------------------------------------------------------------------------
sel = admin_df[(admin_df["network"] == PRIMARY_NET) & (admin_df["year_quarter"] == LATEST_Q)]
m = adm1.merge(sel[["admin1", "digital_divide_index", "median_download_mbps"]],
               on="admin1", how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.4))
cmap_g = mpl.colors.LinearSegmentedColormap.from_list("afdb_g", RAMP)
cmap_r = mpl.colors.LinearSegmentedColormap.from_list(
    "afdb_r", [AFDB["mint"], AFDB["gold"], AFDB["terra"], AFDB["brick"]])

for ax, col, cmap, lbl, kick, ttl in [
    (axes[0], "median_download_mbps", cmap_g, "Median speed (Mbps)", "08 · PERFORMANCE",
     "Median download speed by region"),
    (axes[1], "digital_divide_index", cmap_r, "Index (0–100)", "08 · EQUITY",
     "Digital divide index"),
]:
    m.plot(column=col, cmap=cmap, ax=ax, edgecolor="white", linewidth=0.7, legend=True,
           legend_kwds={"shrink": 0.68, "label": lbl},
           missing_kwds={"color": AFDB["sage"], "label": "No data"})
    ax.set_axis_off()
    ax.text(0, 1.06, kick, transform=ax.transAxes, fontsize=9, fontweight="bold", color=AFDB["deep"])
    ax.text(0, 1.005, ttl, transform=ax.transAxes, fontsize=13.5, fontweight="bold", color=AFDB["ink"])

fig.text(0.01, 0.015, f"Sources: Ookla Open Data {LATEST_Q} ({PRIMARY_NET} network), "
         f"WorldPop {WORLDPOP_YEAR}, geoBoundaries ADM1. Composite index: author's calculation.",
         fontsize=8.5, style="italic", color=AFDB["slate"])
plt.tight_layout(rect=[0, 0.035, 1, 1]); plt.show()

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">09 &middot; KIBANA</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Generating a 13-panel dashboard</div>
</div>

A Kibana dashboard is a **saved object**, i.e. JSON. It can therefore be produced from code, versioned in Git, and replayed identically on another country or another cluster.

**Two design decisions are worth explaining.**

*Why agg-based visualizations rather than Lens.* Lens's internal JSON schema evolves between minor Kibana releases&nbsp;; an object generated for 8.15 may be rejected elsewhere. The `visualization` + `visState` format has been stable since Kibana&nbsp;6 and is still supported. The dashboard is therefore reproducible on a cluster whose version you do not control &mdash; and every panel offers a *Convert to Lens* button if you later want to edit it in the modern editor.

*Why the data views now declare a time field.* With a single quarter we had to disable the time filter to keep panels from turning up empty. With a series, `period` becomes the **time field** of both data views&nbsp;: the picker at the top of the dashboard then filters every panel, and the dashboard stores its own default range (`timeRestore`) so that it opens on actual data.

In [ ]:
# -----------------------------------------------------------------------------
#  9.1  Data views, with a time field
# -----------------------------------------------------------------------------
KB_H = {"kbn-xsrf": "true", "Content-Type": "application/json"}
DV_TILES, DV_ADMIN = f"dv-{INDEX_TILES}", f"dv-{INDEX_ADMIN}"

def create_data_view(dv_id, index, name, time_field="period"):
    r = requests.post(f"{KIBANA_URL}/api/data_views/data_view", headers=KB_H,
                      json={"override": True,
                            "data_view": {"id": dv_id, "title": index, "name": name,
                                          "timeFieldName": time_field}},
                      timeout=120)
    ok = r.status_code in (200, 201)
    print(f"  data view {name:<30} {'OK' if ok else 'FAILED ' + r.text[:200]}")
    return ok

assert service_up(KIBANA_URL, "/api/status"), "Kibana is not reachable."
create_data_view(DV_TILES, INDEX_TILES, f"Ookla · tiles · {ISO}")
create_data_view(DV_ADMIN, INDEX_ADMIN, f"Ookla · regions · {ISO}")

In [ ]:
# -----------------------------------------------------------------------------
#  9.2  Agg-based visualization factories
#
#  One object = attributes.visState (the definition) + kibanaSavedObjectMeta
#  (the data source, referenced against a data view) + uiStateJSON
#  (per-series colours: this is where the AfDB palette is applied).
# -----------------------------------------------------------------------------
def vis_object(so_id, title, vis_state, dv_id, kuery="", colors=None):
    return {
        "id": so_id, "type": "visualization",
        "attributes": {
            "title": title, "description": "",
            "visState": json.dumps(vis_state, ensure_ascii=False),
            "uiStateJSON": json.dumps({"vis": {"colors": colors}} if colors else {}),
            "kibanaSavedObjectMeta": {"searchSourceJSON": json.dumps({
                "query": {"language": "kuery", "query": kuery}, "filter": [],
                "indexRefName": "kibanaSavedObjectMeta.searchSourceJSON.index"})},
        },
        "references": [{"name": "kibanaSavedObjectMeta.searchSourceJSON.index",
                        "type": "index-pattern", "id": dv_id}],
    }

def agg_metric(idx, agg_type, field, label):
    params = {"customLabel": label}
    if agg_type != "count":
        params["field"] = field
    if agg_type == "median":
        params["percents"] = [50]
    return {"id": str(idx), "enabled": True, "type": agg_type,
            "schema": "metric", "params": params}

def agg_terms(idx, field, size, order_by="1", order="desc", label=None):
    return {"id": str(idx), "enabled": True, "type": "terms", "schema": "segment",
            "params": {"field": field, "orderBy": order_by, "order": order, "size": size,
                       "otherBucket": False, "otherBucketLabel": "Other",
                       "missingBucket": False, "missingBucketLabel": "N/A",
                       "customLabel": label or ""}}

def agg_group(idx, field, size=5):
    d = agg_terms(idx, field, size, order_by="_key", order="asc")
    d["schema"] = "group"
    return d

def vs_metric(label, agg_type, field, font=52):
    return {"title": label, "type": "metric",
            "aggs": [agg_metric(1, agg_type, field, label)],
            "params": {"addTooltip": True, "addLegend": False, "type": "metric",
                       "metric": {"percentageMode": False, "useRanges": False,
                                  "colorSchema": "Green to Red", "metricColorMode": "None",
                                  "colorsRange": [{"from": 0, "to": 10000}],
                                  "labels": {"show": True}, "invertColors": False,
                                  "style": {"bgFill": "#000", "bgColor": False,
                                            "labelColor": False, "subText": "",
                                            "fontSize": font}}}}

def _axes(kind, label, horizontal):
    return {
        "type": kind, "grid": {"categoryLines": False},
        "categoryAxes": [{"id": "CategoryAxis-1", "type": "category",
                          "position": "left" if horizontal else "bottom", "show": True,
                          "style": {}, "scale": {"type": "linear"},
                          "labels": {"show": True, "filter": False, "truncate": 120,
                                     "rotate": 0}, "title": {}}],
        "valueAxes": [{"id": "ValueAxis-1", "name": "LeftAxis-1", "type": "value",
                       "position": "bottom" if horizontal else "left", "show": True,
                       "style": {}, "scale": {"type": "linear", "mode": "normal"},
                       "labels": {"show": True, "rotate": 0, "filter": True, "truncate": 100},
                       "title": {"text": label}}],
        "addTooltip": True, "addLegend": False, "legendPosition": "right",
        "times": [], "addTimeMarker": False,
        "labels": {"show": True}, "thresholdLine": {"show": False, "value": 10, "width": 1,
                                                    "style": "full", "color": "#E7664C"},
        "palette": {"type": "palette", "name": "default"},
    }

def vs_bar(label, metric_agg, metric_field, bucket_field, size, horizontal=True,
           order="desc", order_by="1"):
    kind = "horizontal_bar" if horizontal else "histogram"
    p = _axes(kind, label, horizontal)
    p["seriesParams"] = [{"show": True, "type": "histogram", "mode": "normal",
                          "data": {"label": label, "id": "1"}, "valueAxis": "ValueAxis-1",
                          "drawLinesBetweenPoints": True, "lineWidth": 2, "showCircles": True}]
    return {"title": label, "type": kind,
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, bucket_field, size, order_by=order_by, order=order)],
            "params": p}

def vs_line(label, metric_agg, metric_field, x_field, split_field, size=24):
    p = _axes("line", label, horizontal=False)
    p["addLegend"] = True
    p["seriesParams"] = [{"show": True, "type": "line", "mode": "normal",
                          "data": {"label": label, "id": "1"}, "valueAxis": "ValueAxis-1",
                          "drawLinesBetweenPoints": True, "lineWidth": 3,
                          "interpolate": "linear", "showCircles": True}]
    return {"title": label, "type": "line",
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, x_field, size, order_by="_key", order="asc"),
                     agg_group(3, split_field)],
            "params": p}

def vs_pie(label, metric_agg, metric_field, bucket_field, size=6):
    return {"title": label, "type": "pie",
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, bucket_field, size, order_by="_key", order="asc")],
            "params": {"type": "pie", "addTooltip": True, "addLegend": True,
                       "legendPosition": "right", "isDonut": True, "distinctColors": True,
                       "labels": {"show": True, "values": True, "last_level": True,
                                  "truncate": 100, "position": "default",
                                  "valuesFormat": "percent"},
                       "palette": {"type": "palette", "name": "default"}}}

def vs_table(bucket_field, size, metrics):
    """metrics = [(agg_type, field, label), ...]"""
    aggs = [agg_terms(1, bucket_field, size, order_by="2", order="desc", label="Region")]
    for i, (a, f, l) in enumerate(metrics, start=2):
        aggs.append(agg_metric(i, a, f, l))
    return {"title": "Summary", "type": "table", "aggs": aggs,
            "params": {"perPage": 12, "showPartialRows": False, "showTotal": False,
                       "showMetricsAtAllLevels": False, "totalFunc": "sum",
                       "percentageCol": "", "showToolbar": True,
                       "palette": {"type": "palette", "name": "default"}}}

print("Visualization factories ready.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.3  The twelve analytical panels
# -----------------------------------------------------------------------------
P = f"ookla-{ISO.lower()}"
NET_Q = f'network : "{PRIMARY_NET}"'
objects = []

# --- Four headline metrics ----------------------------------------------------
objects += [
    vis_object(f"{P}-kpi-tests", "Speed tests",
               vs_metric("Speed tests", "sum", "tests"), DV_TILES),
    vis_object(f"{P}-kpi-dl", f"Median download ({PRIMARY_NET})",
               vs_metric(f"Median download {PRIMARY_NET} (Mbps)", "median", "download_mbps"),
               DV_TILES, kuery=NET_Q),
    vis_object(f"{P}-kpi-pop", "Measured population",
               vs_metric("Population under measured tiles", "sum", "population"),
               DV_TILES, kuery=NET_Q),
    vis_object(f"{P}-kpi-lat", "Median latency",
               vs_metric("Median latency (ms)", "median", "latency_ms"),
               DV_TILES, kuery=NET_Q),
]

# --- Time series --------------------------------------------------------------
objects.append(vis_object(
    f"{P}-line-evo", "Median download speed by quarter",
    vs_line("Median download (Mbps)", "median", "download_mbps", "year_quarter", "network"),
    DV_TILES, colors={"fixed": AFDB["green"], "mobile": AFDB["teal"]}))

# --- Population split by speed class ------------------------------------------
objects.append(vis_object(
    f"{P}-pie-speed", "Population by speed class",
    vs_pie("People", "sum", "population", "speed_class"),
    DV_TILES, kuery=NET_Q,
    colors={SPEED_CLASSES[0][2]: AFDB["brick"], SPEED_CLASSES[1][2]: AFDB["ochre"],
            SPEED_CLASSES[2][2]: RAMP[3],       SPEED_CLASSES[3][2]: AFDB["deep"]}))

# --- Regional rankings --------------------------------------------------------
objects += [
    vis_object(f"{P}-bar-dl", "Median download by region",
               vs_bar("Median download (Mbps)", "median", "download_mbps", "admin1", 12),
               DV_TILES, kuery=NET_Q, colors={"Median download (Mbps)": AFDB["green"]}),
    vis_object(f"{P}-bar-ddi", "Digital divide index",
               vs_bar("Digital divide index", "max", "digital_divide_index", "admin1", 12),
               DV_ADMIN, kuery=NET_Q, colors={"Digital divide index": AFDB["terra"]}),
    vis_object(f"{P}-bar-settlement", "Median download by settlement type",
               vs_bar("Median download (Mbps)", "median", "download_mbps", "settlement", 5,
                      horizontal=False, order="asc", order_by="_key"),
               DV_TILES, kuery=NET_Q, colors={"Median download (Mbps)": AFDB["deep"]}),
    vis_object(f"{P}-bar-lat", "Median latency by region",
               vs_bar("Median latency (ms)", "median", "latency_ms", "admin1", 12),
               DV_TILES, kuery=NET_Q, colors={"Median latency (ms)": AFDB["ochre"]}),
]

# --- Summary table ------------------------------------------------------------
objects.append(vis_object(
    f"{P}-table", "Regional summary",
    vs_table("admin1", 40, [("max", "population", "Population"),
                            ("max", "median_download_mbps", "Median download (Mbps)"),
                            ("max", "pct_pop_10mbps", "% pop. ≥ 10 Mbps"),
                            ("max", "median_latency_ms", "Latency (ms)"),
                            ("max", "digital_divide_index", "Digital divide index")]),
    DV_ADMIN, kuery=NET_Q))

print(f"{len(objects)} visualizations generated.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.4  Maps object: tiles (MVT) + regional outlines
# -----------------------------------------------------------------------------
center_lon, center_lat = (BBOX[0] + BBOX[2]) / 2, (BBOX[1] + BBOX[3]) / 2
span = max(BBOX[2] - BBOX[0], BBOX[3] - BBOX[1])
map_zoom = int(max(4, min(9, math.floor(math.log2(360.0 / max(span, 0.5))) + 1)))

basemap_layer = {
    "id": "lyr_base", "label": None, "minZoom": 0, "maxZoom": 24, "alpha": 1,
    "visible": True, "type": "EMS_VECTOR_TILE", "includeInFitToBounds": True,
    "style": {"type": "TILE"},
    "sourceDescriptor": {"type": "EMS_TMS", "isAutoSelect": True,
                         "lightModeDefault": "road_map_desaturated"},
}

tiles_layer = {
    "id": "lyr_tiles", "label": f"Ookla tiles — download speed ({PRIMARY_NET})",
    "minZoom": 0, "maxZoom": 24, "alpha": 0.85, "visible": True,
    "type": "MVT_VECTOR", "joins": [], "includeInFitToBounds": True,
    "query": {"language": "kuery", "query": NET_Q},
    "sourceDescriptor": {
        "type": "ES_SEARCH", "id": "src_tiles",
        "indexPatternRefName": "layer_2_source_index_pattern",
        "geoField": "tile_geom", "scalingType": "MVT", "filterByMapBounds": True,
        "tooltipProperties": ["quadkey", "admin1", "year_quarter", "download_mbps",
                              "upload_mbps", "latency_ms", "tests", "population"],
        "sortField": "", "sortOrder": "desc", "applyGlobalQuery": True,
        "applyGlobalTime": True, "applyForceRefresh": True,
    },
    "style": {"type": "VECTOR", "isTimeAware": True, "properties": {
        "fillColor": {"type": "DYNAMIC", "options": {
            "color": "Greens", "colorCategory": "palette_0", "type": "ORDINAL",
            "field": {"name": "download_mbps", "origin": "source"},
            "fieldMetaOptions": {"isEnabled": True, "sigma": 3}}},
        "lineColor": {"type": "STATIC", "options": {"color": "#FFFFFF"}},
        "lineWidth": {"type": "STATIC", "options": {"size": 0}},
        "iconSize": {"type": "STATIC", "options": {"size": 6}},
        "icon": {"type": "STATIC", "options": {"value": "marker"}},
        "symbolizeAs": {"options": {"value": "circle"}},
        "iconOrientation": {"type": "STATIC", "options": {"orientation": 0}},
        "labelText": {"type": "STATIC", "options": {"value": ""}},
        "labelColor": {"type": "STATIC", "options": {"color": "#000000"}},
        "labelSize": {"type": "STATIC", "options": {"size": 14}},
        "labelBorderColor": {"type": "STATIC", "options": {"color": "#FFFFFF"}},
        "labelBorderSize": {"options": {"size": "SMALL"}},
    }},
}

admin_layer = {
    "id": "lyr_adm", "label": "Regions (ADM1)", "minZoom": 0, "maxZoom": 24,
    "alpha": 1, "visible": True, "type": "GEOJSON_VECTOR", "joins": [],
    "includeInFitToBounds": True,
    "query": {"language": "kuery", "query": NET_Q},
    "sourceDescriptor": {
        "type": "ES_SEARCH", "id": "src_adm",
        "indexPatternRefName": "layer_3_source_index_pattern",
        "geoField": "geometry", "scalingType": "LIMIT", "topHitsSize": 1,
        "filterByMapBounds": False, "applyGlobalQuery": True, "applyGlobalTime": True,
        "sortField": "", "sortOrder": "desc",
        "tooltipProperties": ["admin1", "year_quarter", "population",
                              "median_download_mbps", "pct_pop_10mbps",
                              "digital_divide_index"],
    },
    "style": {"type": "VECTOR", "properties": {
        "fillColor": {"type": "STATIC", "options": {"color": "rgba(0,0,0,0)"}},
        "lineColor": {"type": "STATIC", "options": {"color": AFDB["deep"]}},
        "lineWidth": {"type": "STATIC", "options": {"size": 1.6}},
        "iconSize": {"type": "STATIC", "options": {"size": 6}},
        "symbolizeAs": {"options": {"value": "circle"}},
        "labelText": {"type": "STATIC", "options": {"value": ""}},
    }},
}

objects.append({
    "id": f"{P}-map", "type": "map",
    "attributes": {
        "title": f"Connectivity map — {COUNTRY_NAME}",
        "description": "Ookla tiles as geo_shape (MVT) and ADM1 outlines.",
        "layerListJSON": json.dumps([basemap_layer, admin_layer, tiles_layer]),
        "mapStateJSON": json.dumps({
            "zoom": map_zoom, "center": {"lon": center_lon, "lat": center_lat},
            "timeFilters": {"from": "now-5y", "to": "now"},
            "refreshConfig": {"isPaused": True, "interval": 0},
            "query": {"language": "kuery", "query": ""}, "filters": [],
            "settings": {"autoFitToDataBounds": False, "backgroundColor": "#ffffff",
                         "disableInteractive": False, "disableTooltipControl": False,
                         "hideToolbarOverlay": False, "hideLayerControl": False,
                         "hideViewControl": False, "initialLocation": "LAST_SAVED_LOCATION",
                         "showScaleControl": True, "showSpatialFilters": True,
                         "spatialFiltersAlpa": 0.3}}),
        "uiStateJSON": json.dumps({"isLayerTOCOpen": False, "openTOCDetails": []}),
    },
    "references": [
        {"name": "layer_2_source_index_pattern", "type": "index-pattern", "id": DV_TILES},
        {"name": "layer_3_source_index_pattern", "type": "index-pattern", "id": DV_ADMIN},
    ],
})
print(f"Map defined — centre ({center_lat:.2f}, {center_lon:.2f}), zoom {map_zoom}.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.5  Dashboard assembly (48-column grid) + stored time range
# -----------------------------------------------------------------------------
HEADER_MD = (
    f"## {COUNTRY_NAME} — Ookla connectivity x population\n"
    f"**{' → '.join(QUARTER_LABELS)}** · *{PRIMARY_NET}* network · "
    f"{len(tiles):,} tiles of ~610 m indexed as `geo_shape`\n\n"
    f"Use the time picker in the top right to filter every panel at once. "
    f"Sources: Ookla Open Data · WorldPop {WORLDPOP_YEAR} (1 km UN-adjusted) · "
    f"geoBoundaries ADM1. The digital divide index is a teaching composite."
).replace(",", " ")

LAYOUT = [
    (f"{P}-kpi-tests",      "Speed tests",                        0,  6, 12,  8),
    (f"{P}-kpi-dl",         f"Median download ({PRIMARY_NET})",  12,  6, 12,  8),
    (f"{P}-kpi-pop",        "Measured population",               24,  6, 12,  8),
    (f"{P}-kpi-lat",        "Median latency",                    36,  6, 12,  8),
    (f"{P}-line-evo",       "Quarterly speed evolution",          0, 14, 48, 14),
    (f"{P}-map",            "Connectivity map",                   0, 28, 28, 19),
    (f"{P}-pie-speed",      "Population by speed class",         28, 28, 20, 19),
    (f"{P}-bar-dl",         "Median download by region",          0, 47, 24, 16),
    (f"{P}-bar-ddi",        "Digital divide index",              24, 47, 24, 16),
    (f"{P}-bar-settlement", "Speed by settlement type",           0, 63, 20, 15),
    (f"{P}-bar-lat",        "Median latency by region",          20, 63, 28, 15),
    (f"{P}-table",          "Regional summary",                   0, 78, 48, 18),
]

panels, refs = [], []
panels.append({                       # markdown header, embedded "by value"
    "version": ES_VERSION, "type": "visualization", "panelIndex": "0",
    "gridData": {"x": 0, "y": 0, "w": 48, "h": 6, "i": "0"},
    "embeddableConfig": {"savedVis": {
        "id": "", "title": "", "description": "", "type": "markdown",
        "params": {"fontSize": 11, "openLinksInNewTab": True, "markdown": HEADER_MD},
        "uiState": {}, "data": {"aggs": [], "searchSource": {}}},
        "hidePanelTitles": True, "enhancements": {}},
})

known = {o["id"] for o in objects}
for i, (so_id, title, x, y, w, h) in enumerate(LAYOUT, start=1):
    if so_id not in known:
        continue
    so_type = "map" if so_id.endswith("-map") else "visualization"
    panels.append({"version": ES_VERSION, "type": so_type, "panelIndex": str(i),
                   "gridData": {"x": x, "y": y, "w": w, "h": h, "i": str(i)},
                   "embeddableConfig": {"enhancements": {}},
                   "panelRefName": f"panel_{i}", "title": title})
    refs.append({"name": f"panel_{i}", "type": so_type, "id": so_id})

# Stored time range: from the start of the first quarter to now.
first_period = min(tiles["period"])
dashboard = {
    "id": f"{P}-dashboard", "type": "dashboard",
    "attributes": {
        "title": DASHBOARD_TITLE,
        "description": (f"Generated by notebook — {COUNTRY_NAME}, {', '.join(QUARTER_LABELS)}, "
                        f"WorldPop {WORLDPOP_YEAR}. AfDB-inspired Institutional style."),
        "panelsJSON": json.dumps(panels),
        "optionsJSON": json.dumps({"useMargins": True, "syncColors": False,
                                   "syncCursor": True, "syncTooltips": False,
                                   "hidePanelTitles": False}),
        "timeRestore": True,
        "timeFrom": f"{first_period}T00:00:00.000Z",
        "timeTo": "now",
        "refreshInterval": {"pause": True, "value": 0},
        "version": 1,
        "kibanaSavedObjectMeta": {"searchSourceJSON": json.dumps(
            {"query": {"language": "kuery", "query": ""}, "filter": []})},
    },
    "references": refs,
}
print(f"Dashboard assembled: {len(panels)} panels, range {first_period} → now.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.6  Object-by-object import (+ reusable NDJSON)
#
#  Each visualization is imported individually, then the dashboard is assembled
#  only from the panels that were actually accepted: a rejected object no longer
#  takes the whole dashboard down with it.
# -----------------------------------------------------------------------------
def kb_import(objs):
    nd = "\n".join(json.dumps(o, ensure_ascii=False) for o in objs)
    r = requests.post(f"{KIBANA_URL}/api/saved_objects/_import?overwrite=true",
                      headers={"kbn-xsrf": "true"},
                      files={"file": ("o.ndjson", nd, "application/ndjson")}, timeout=180)
    try:
        return r.json()
    except Exception:
        return {"success": False, "errors": [{"http": r.status_code, "body": r.text[:400]}]}

NDJSON_PATH = OUTDIR / f"dashboard_ookla_{ISO.lower()}.ndjson"
NDJSON_PATH.write_text("\n".join(json.dumps(o, ensure_ascii=False) for o in objects + [dashboard]),
                       encoding="utf-8")

ok_ids, failures = [], []
for o in objects:
    res = kb_import([o])
    if res.get("successCount"):
        ok_ids.append(o["id"])
    else:
        failures.append((o["id"], res.get("errors")))

keep = {r["name"] for r in refs if r["id"] in ok_ids}
dash_final = json.loads(json.dumps(dashboard))
dash_final["attributes"]["panelsJSON"] = json.dumps(
    [p for p in panels if p.get("panelRefName") is None or p["panelRefName"] in keep])
dash_final["references"] = [r for r in refs if r["id"] in ok_ids]
res_dash = kb_import([dash_final])

kpi_row([
    (f"{len(ok_ids)}/{len(objects)}", "Visualizations imported", AFDB["green"]),
    ("YES" if res_dash.get("successCount") else "NO", "Dashboard created",
     AFDB["deep"] if res_dash.get("successCount") else AFDB["brick"]),
    (f"{len(failures)}", "Objects rejected", AFDB["brick"] if failures else AFDB["slate"]),
])
for oid, err in failures:
    print(f"\nRejected: {oid}\n  " + json.dumps(err, ensure_ascii=False)[:420])
if res_dash.get("errors"):
    print("\nDashboard error:", json.dumps(res_dash["errors"], ensure_ascii=False)[:500])
print(f"\nReusable NDJSON: {NDJSON_PATH}")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">10 &middot; ACCESS</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Opening the dashboard</div>
</div>

In [ ]:
# -----------------------------------------------------------------------------
#  10.1  Direct link to the dashboard
# -----------------------------------------------------------------------------
base = kibana_public_url(open_window=False) or KIBANA_PUBLIC
url = (base.rstrip("/") + f"/app/dashboards#/view/{P}-dashboard") if base else None

display(HTML(f"""
<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            border-radius:14px;padding:26px 30px;font-family:Calibri,'Segoe UI',sans-serif;color:#fff;">
  <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#F5C242;text-transform:uppercase;">DASHBOARD READY</div>
  <div style="font-size:26px;font-weight:700;margin-top:8px;">{DASHBOARD_TITLE}</div>
  <div style="font-size:14px;color:#E6F6EE;margin-top:10px;font-style:italic;">
    {('<a href="' + url + '" target="_blank" style="color:#F5C242;font-weight:700;">Open in Kibana &rarr;</a>') if url else 'Use a tunnel, or import the NDJSON into your own Kibana.'}
  </div>
  <div style="height:5px;width:110px;background:#F5C242;border-radius:3px;margin-top:22px;"></div>
</div>"""))
if url:
    print(url)
    if IN_COLAB:
        try:
            from google.colab.output import serve_kernel_port_as_window
            serve_kernel_port_as_window(KIBANA_PORT)
        except Exception:
            pass

# --- Verification: what actually exists on the Kibana side --------------------
try:
    found = requests.get(f"{KIBANA_URL}/api/saved_objects/_find",
                         params={"type": ["dashboard", "visualization", "map"],
                                 "fields": "title", "per_page": 100},
                         headers={"kbn-xsrf": "true"}, timeout=60).json()
    objs = found.get("saved_objects", [])
    print(f"\n{len(objs)} saved objects present:")
    for o in sorted(objs, key=lambda x: (x["type"], x["id"])):
        print(f"  {o['type']:<14} {o['id']:<30} {o['attributes'].get('title','')}")
except Exception as exc:
    print("Verification failed:", exc)

In [ ]:
# -----------------------------------------------------------------------------
#  10.2  Additional exports (sharing outside Kibana)
# -----------------------------------------------------------------------------
tiles_out = OUTDIR / f"ookla_tiles_{ISO.lower()}.parquet"
admin_out = OUTDIR / f"ookla_admin_{ISO.lower()}.csv"
tiles.to_parquet(tiles_out, index=False)
admin_df.to_csv(admin_out, index=False)

files = [(NDJSON_PATH, "Kibana saved objects (dashboard + visualizations + map)"),
         (tiles_out,   "Enriched tiles, all quarters (Parquet)"),
         (admin_out,   "Region x network x quarter summary (CSV)")]
rows = "".join(f"<tr><td style='padding:6px 12px;font-family:monospace;font-size:12px;'>{p.name}</td>"
               f"<td style='padding:6px 12px;font-size:13px;color:{AFDB['slate']};'>{d}</td>"
               f"<td style='padding:6px 12px;font-size:12px;text-align:right;'>{p.stat().st_size/1e6:,.2f} MB</td></tr>"
               for p, d in files)
display(HTML(f"<table style=\"border-collapse:collapse;border:1px solid {AFDB['sage']};"
             f"border-radius:10px;overflow:hidden;font-family:Calibri,sans-serif;\">"
             f"<thead style='background:{AFDB['mint']};'><tr>"
             f"<th style='padding:8px 12px;text-align:left;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>FILE</th>"
             f"<th style='padding:8px 12px;text-align:left;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>CONTENT</th>"
             f"<th style='padding:8px 12px;text-align:right;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>SIZE</th>"
             f"</tr></thead><tbody>{rows}</tbody></table>"))
print("Output directory:", OUTDIR)

<div style="border-left:5px solid #F5C242;background:#FDF4E0;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#D49A00;text-transform:uppercase;">GOING FURTHER</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Directions for part 2</div>
</div>

* **Data streams and ILM** &mdash; replace the single index with a *data stream* and a lifecycle policy&nbsp;: quarterly rollover, move to *warm* after a year.
* **Runtime fields** &mdash; compute the divide index on the fly in the cluster (Painless) instead of freezing it at index time&nbsp;: the weights become adjustable without reindexing.
* **Transforms** &mdash; the `_transform` API builds the regional aggregation index without Python, and keeps it continuously up to date.
* **Alerting** &mdash; a Kibana rule on quarter-over-quarter degradation of a region's median speed.
* **Richer joins** &mdash; cross with schools and health facilities (OpenStreetMap, Healthsites.io) through `geo_distance` to measure public infrastructure connectivity.
* **Vector tiles** &mdash; the `_mvt` endpoint serves vector tiles directly to your own mapping application.

In [ ]:
# -----------------------------------------------------------------------------
#  Cleanup (uncomment to free the machine)
# -----------------------------------------------------------------------------
# es.indices.delete(index=INDEX_TILES, ignore_unavailable=True)
# es.indices.delete(index=INDEX_ADMIN, ignore_unavailable=True)
# sh("pkill -f elasticsearch; pkill -f kibana")
# shutil.rmtree(STACKDIR, ignore_errors=True)

card("Session complete",
     f"Cluster <b>{info['cluster_name']}</b> · indices <code>{INDEX_TILES}</code> and "
     f"<code>{INDEX_ADMIN}</code> · {len(QUARTER_LABELS)} quarters · dashboard "
     f"<b>{DASHBOARD_TITLE}</b>.<br>"
     "The cleanup cells are commented out on purpose&nbsp;: explore Kibana first, "
     "then free the resources.", tone="ok")